# Phase 1: Data Diagnostics & Intelligence (EDA)

**Objective:** Analyze dataset quality before YOLO conversion to prevent "learning noise."

---

## Table of Contents

1. [Configuration & Setup](#config)
2. [Load & Parse COCO JSON](#load)
3. [Dataset Overview Dashboard](#overview)
4. [Image Dimension Analysis](#dimensions)
5. [Area & Aspect Ratio Analysis](#area-ar)
6. [Spatial Distribution Heatmap](#heatmap)
7. [Hierarchical Consistency Check](#hierarchy)
8. [Overlap / IoU Analysis](#iou)
9. [Negative Sample Strategy](#negatives)
10. [Outlier Sample Visualization](#outlier-viz)
11. [Summary Report & Recommendations](#summary)

---

<a id='config'></a>
## 1. Configuration & Setup

In [ ]:
import json
import os
import warnings
from pathlib import Path
from collections import defaultdict
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Rectangle
import seaborn as sns
from PIL import Image

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# -------------------- PATHS --------------------
JSON_PATH = '/kaggle/input/datasets/mohammedahmedxx12/machathon-dataset/Phase1_Train_Dataset/annotations/Cells_Anotations_coco.json'
IMAGES_DIR = '/kaggle/input/datasets/mohammedahmedxx12/machathon-dataset/Phase1_Train_Dataset/images'
OUTPUT_DIR = '/kaggle/working'

# -------------------- THRESHOLDS --------------------
TINY_AREA_PCT = 0.03        # Boxes < 3% of image area are flagged (for tables only)
FULL_PAGE_PCT = 0.95        # Boxes > 95% of image area are flagged
EXTREME_AR_LOW = 0.2        # Aspect ratio < 0.2 (extremely tall)
EXTREME_AR_HIGH = 5.0       # Aspect ratio > 5.0 (extremely wide)
NEGATIVE_RATIO_MIN = 0.10   # Target 10% minimum empty images
NEGATIVE_RATIO_MAX = 0.15   # Target 15% maximum empty images
IOU_DUPLICATE_THRESH = 0.8  # IoU > 0.8 between same-category boxes = duplicate

# -------------------- CATEGORY MAPPING --------------------
# Will be populated after loading JSON
CAT_ID_TO_NAME = {}
CAT_NAME_TO_ID = {}

# -------------------- COLOR PALETTE --------------------
CATEGORY_COLORS = {
    1: '#E74C3C',  # table - red
    2: '#3498DB',  # table column - blue
    3: '#2ECC71',  # table row - green
    4: '#9B59B6',  # table column header - purple
    5: '#F39C12',  # table projected row header - orange
    6: '#1ABC9C',  # table spanning cell - teal
}

# Seaborn style
sns.set_theme(style='whitegrid', palette='husl')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print("loaded")

<a id='load'></a>
## 2. Load & Parse COCO JSON

In [ ]:
with open(JSON_PATH, 'r') as f:
    coco_data = json.load(f)

# Extract components
images_list = coco_data['images']
annotations_list = coco_data['annotations']
categories_list = coco_data['categories']

print(f"   Loaded {len(images_list):,} images")
print(f"   Loaded {len(annotations_list):,} annotations")
print(f"   Loaded {len(categories_list)} categories")

# -------------------- Build DataFrames --------------------

# Categories DataFrame
df_cats = pd.DataFrame(categories_list)
CAT_ID_TO_NAME = dict(zip(df_cats['id'], df_cats['name']))
CAT_NAME_TO_ID = dict(zip(df_cats['name'], df_cats['id']))

print("\n Categories:")
for _, row in df_cats.iterrows():
    print(f"   {row['id']}: {row['name']} (parent: {row['supercategory']})")

# Images DataFrame
df_images = pd.DataFrame(images_list)
df_images['image_area'] = df_images['width'] * df_images['height']
df_images = df_images.rename(columns={'id': 'image_id'})

# Annotations DataFrame
df_anns = pd.DataFrame(annotations_list)

# Decode bbox [x, y, width, height] into separate columns
df_anns['bbox_x'] = df_anns['bbox'].apply(lambda b: b[0])
df_anns['bbox_y'] = df_anns['bbox'].apply(lambda b: b[1])
df_anns['bbox_w'] = df_anns['bbox'].apply(lambda b: b[2])
df_anns['bbox_h'] = df_anns['bbox'].apply(lambda b: b[3])

# Compute center coordinates
df_anns['bbox_cx'] = df_anns['bbox_x'] + df_anns['bbox_w'] / 2
df_anns['bbox_cy'] = df_anns['bbox_y'] + df_anns['bbox_h'] / 2

# -------------------- Merge with Images --------------------
df_anns = df_anns.merge(df_images[['image_id', 'file_name', 'width', 'height', 'image_area']], 
                        on='image_id', how='left')

# -------------------- Derived Columns --------------------
df_anns['box_area_pct'] = df_anns['area'] / df_anns['image_area']
df_anns['aspect_ratio'] = df_anns['bbox_w'] / df_anns['bbox_h'].replace(0, np.nan)
df_anns['category_name'] = df_anns['category_id'].map(CAT_ID_TO_NAME)

# Normalized center (0-1 range)
df_anns['norm_cx'] = df_anns['bbox_cx'] / df_anns['width']
df_anns['norm_cy'] = df_anns['bbox_cy'] / df_anns['height']

print("\n DataFrames built successfully.")
print(f"   df_images shape: {df_images.shape}")
print(f"   df_anns shape: {df_anns.shape}")
print(f"   df_cats shape: {df_cats.shape}")

# Quick sanity check
print("\n Sample annotation record:")
display(df_anns.head(1).T)

<a id='overview'></a>
## 3. Dataset Overview Dashboard

In [ ]:
# ============================================================
# CELL 2: Dataset Overview Dashboard
# ============================================================

print("="*60)
print(" DATASET OVERVIEW")
print("="*60)

n_images = len(df_images)
n_annotations = len(df_anns)
n_categories = len(df_cats)

print(f"\n Total Images: {n_images:,}")
print(f" Total Annotations: {n_annotations:,}")
print(f" Avg Annotations/Image: {n_annotations/n_images:.1f}")
print(f" Categories: {n_categories}")

# -------------------- Annotation Count per Category --------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart: Annotations per category
cat_counts = df_anns['category_name'].value_counts().sort_index()
colors = [CATEGORY_COLORS.get(CAT_NAME_TO_ID.get(name, 1), '#333333') for name in cat_counts.index]

ax1 = axes[0]
bars = ax1.bar(cat_counts.index, cat_counts.values, color=colors, edgecolor='black', linewidth=0.5)
ax1.set_xlabel('Category', fontsize=12)
ax1.set_ylabel('Annotation Count', fontsize=12)
ax1.set_title('Annotation Count per Category (Class Imbalance Check)', fontsize=13, fontweight='bold')
ax1.tick_params(axis='x', rotation=45)

# Add count labels on bars
for bar, count in zip(bars, cat_counts.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100, 
             f'{count:,}', ha='center', va='bottom', fontsize=10)

# -------------------- Annotations per Image Distribution --------------------
anns_per_image = df_anns.groupby('image_id').size()

ax2 = axes[1]
ax2.hist(anns_per_image, bins=50, color='#3498DB', edgecolor='black', alpha=0.7)
ax2.axvline(anns_per_image.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {anns_per_image.mean():.1f}')
ax2.axvline(anns_per_image.median(), color='orange', linestyle='--', linewidth=2, label=f'Median: {anns_per_image.median():.1f}')
ax2.set_xlabel('Annotations per Image', fontsize=12)
ax2.set_ylabel('Frequency', fontsize=12)
ax2.set_title('Distribution of Annotations per Image', fontsize=13, fontweight='bold')
ax2.legend()

plt.tight_layout()
plt.show()

# -------------------- Flag Outlier Images --------------------
mean_anns = anns_per_image.mean()
std_anns = anns_per_image.std()
threshold_high = mean_anns + 3 * std_anns
threshold_low = 3  # Images with very few annotations

high_ann_images = anns_per_image[anns_per_image > threshold_high]
low_ann_images = anns_per_image[anns_per_image < threshold_low]

print(f"\n  Outlier Detection (Annotations per Image):")
print(f"   Mean: {mean_anns:.1f}, Std: {std_anns:.1f}")
print(f"   High threshold (>3σ): {threshold_high:.1f}")
print(f"   Images with >3σ annotations: {len(high_ann_images)}")
print(f"   Images with <3 annotations: {len(low_ann_images)}")

if len(high_ann_images) > 0:
    print(f"\n    High annotation count images (top 5):")
    for img_id, count in high_ann_images.nlargest(5).items():
        fname = df_images[df_images['image_id'] == img_id]['file_name'].values[0]
        print(f"      {fname}: {count} annotations")

<a id='dimensions'></a>
## 4. Image Dimension Analysis

In [ ]:
# ============================================================
# CELL 3: Image Dimension Analysis
# ============================================================

print("="*60)
print(" IMAGE DIMENSION ANALYSIS")
print("="*60)

# Check dimension variance
unique_dims = df_images.groupby(['width', 'height']).size().reset_index(name='count')
unique_dims = unique_dims.sort_values('count', ascending=False)

print(f"\n Unique Image Dimensions: {len(unique_dims)}")
print("\n   Top 10 most common dimensions:")
for _, row in unique_dims.head(10).iterrows():
    pct = row['count'] / n_images * 100
    print(f"      {row['width']}x{row['height']}: {row['count']:,} images ({pct:.1f}%)")

# -------------------- Visualization --------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot of dimensions
ax1 = axes[0]
scatter = ax1.scatter(df_images['width'], df_images['height'], 
                      alpha=0.5, c='#3498DB', edgecolors='black', linewidth=0.3, s=50)
ax1.set_xlabel('Width (pixels)', fontsize=12)
ax1.set_ylabel('Height (pixels)', fontsize=12)
ax1.set_title('Image Dimensions Scatter Plot', fontsize=13, fontweight='bold')

# Add most common dimension marker
most_common = unique_dims.iloc[0]
ax1.scatter(most_common['width'], most_common['height'], 
            c='red', s=200, marker='*', zorder=5, label=f"Most common: {most_common['width']}x{most_common['height']}")
ax1.legend()

# Aspect ratio distribution of images
ax2 = axes[1]
img_aspect_ratios = df_images['width'] / df_images['height']
ax2.hist(img_aspect_ratios, bins=30, color='#2ECC71', edgecolor='black', alpha=0.7)
ax2.axvline(img_aspect_ratios.mean(), color='red', linestyle='--', linewidth=2, 
            label=f'Mean AR: {img_aspect_ratios.mean():.2f}')
ax2.set_xlabel('Image Aspect Ratio (W/H)', fontsize=12)
ax2.set_ylabel('Frequency', fontsize=12)
ax2.set_title('Image Aspect Ratio Distribution', fontsize=13, fontweight='bold')
ax2.legend()

plt.tight_layout()
plt.show()

# -------------------- Input Resolution Recommendation --------------------
mean_width = df_images['width'].mean()
mean_height = df_images['height'].mean()
mean_ar = mean_width / mean_height

print(f"\n Image Statistics:")
print(f"   Mean Width: {mean_width:.0f}px")
print(f"   Mean Height: {mean_height:.0f}px")
print(f"   Mean Aspect Ratio: {mean_ar:.3f}")

# Recommendation based on aspect ratio
if 0.9 <= mean_ar <= 1.1:
    rec_res = "640x640 or 1280x1280 (square)"
elif mean_ar < 0.9:
    # Taller images
    rec_res = "640x960 or 640x1280 (portrait)"
else:
    # Wider images
    rec_res = "960x640 or 1280x640 (landscape)"

print(f"\n Recommended YOLOv10 Input Resolution: {rec_res}")

<a id='area-ar'></a>
## 5. Area & Aspect Ratio Analysis (Core Requirement 1)

In [ ]:
# ============================================================
# CELL 4a: Box Area vs. Image Area Analysis
# ============================================================

print("="*60)
print(" AREA ANALYSIS: Box Area % of Image")
print("="*60)

# -------------------- Statistics by Category --------------------
print("\n Box Area Percentage Statistics by Category:")
area_stats = df_anns.groupby('category_name')['box_area_pct'].describe()
display(area_stats.round(4))

# -------------------- Scatter Plot: Box Area % vs Image Area --------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax1 = axes[0]
for cat_id, cat_name in CAT_ID_TO_NAME.items():
    subset = df_anns[df_anns['category_id'] == cat_id]
    ax1.scatter(subset['image_area'], subset['box_area_pct'] * 100, 
                alpha=0.4, label=cat_name, c=CATEGORY_COLORS.get(cat_id, '#333'),
                s=20, edgecolors='none')

# Threshold lines
ax1.axhline(y=TINY_AREA_PCT * 100, color='orange', linestyle='--', linewidth=2, label=f'Tiny threshold ({TINY_AREA_PCT*100}%)')
ax1.axhline(y=FULL_PAGE_PCT * 100, color='red', linestyle='--', linewidth=2, label=f'Full-page threshold ({FULL_PAGE_PCT*100}%)')

ax1.set_xlabel('Image Area (pixels²)', fontsize=12)
ax1.set_ylabel('Box Area (% of Image)', fontsize=12)
ax1.set_title('Box Area % vs Image Area (by Category)', fontsize=13, fontweight='bold')
ax1.legend(loc='upper right', fontsize=8)
ax1.set_ylim(0, 105)

# -------------------- KDE Distribution by Category --------------------
ax2 = axes[1]
for cat_id, cat_name in CAT_ID_TO_NAME.items():
    subset = df_anns[df_anns['category_id'] == cat_id]
    if len(subset) > 10:
        sns.kdeplot(data=subset['box_area_pct'] * 100, ax=ax2, 
                    label=cat_name, color=CATEGORY_COLORS.get(cat_id, '#333'),
                    linewidth=2)

ax2.axvline(x=TINY_AREA_PCT * 100, color='orange', linestyle='--', linewidth=2)
ax2.axvline(x=FULL_PAGE_PCT * 100, color='red', linestyle='--', linewidth=2)
ax2.set_xlabel('Box Area (% of Image)', fontsize=12)
ax2.set_ylabel('Density', fontsize=12)
ax2.set_title('Box Area % Distribution by Category (KDE)', fontsize=13, fontweight='bold')
ax2.legend(loc='upper right', fontsize=9)
ax2.set_xlim(0, 100)

plt.tight_layout()
plt.show()

# -------------------- Flag Tiny Boxes (TABLES ONLY) --------------------
table_cat_id = CAT_NAME_TO_ID.get('table', 1)
df_tiny_tables = df_anns[(df_anns['category_id'] == table_cat_id) & 
                          (df_anns['box_area_pct'] < TINY_AREA_PCT)].copy()

print(f"\n  TINY TABLE BOXES (<{TINY_AREA_PCT*100}% of page):")
print(f"   Found: {len(df_tiny_tables)} annotations")
if len(df_tiny_tables) > 0:
    print("   Sample flagged files:")
    for _, row in df_tiny_tables.head(5).iterrows():
        print(f"      - {row['file_name']} (ann_id: {row['id']}, area%: {row['box_area_pct']*100:.2f}%)")

# -------------------- Flag Full-Page Boxes (ALL CATEGORIES) --------------------
df_fullpage = df_anns[df_anns['box_area_pct'] > FULL_PAGE_PCT].copy()

print(f"\n  FULL-PAGE BOXES (>{FULL_PAGE_PCT*100}% of page):")
print(f"   Found: {len(df_fullpage)} annotations")
if len(df_fullpage) > 0:
    print("   Sample flagged files:")
    for _, row in df_fullpage.head(5).iterrows():
        print(f"      - {row['file_name']} (ann_id: {row['id']}, cat: {row['category_name']}, area%: {row['box_area_pct']*100:.2f}%)")

In [ ]:
# ============================================================
# CELL 4b: Width-to-Height Ratio (Aspect Ratio) Analysis
# ============================================================

print("="*60)
print(" ASPECT RATIO ANALYSIS: Width / Height")
print("="*60)

# -------------------- Statistics by Category --------------------
print("\n Aspect Ratio Statistics by Category:")
ar_stats = df_anns.groupby('category_name')['aspect_ratio'].describe()
display(ar_stats.round(3))

# -------------------- Histogram + Box Plot --------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Histogram by category
ax1 = axes[0]
for cat_id, cat_name in CAT_ID_TO_NAME.items():
    subset = df_anns[df_anns['category_id'] == cat_id]
    # Clip extreme values for visualization
    ar_clipped = subset['aspect_ratio'].clip(0, 10)
    ax1.hist(ar_clipped, bins=50, alpha=0.5, label=cat_name, 
             color=CATEGORY_COLORS.get(cat_id, '#333'))

ax1.axvline(x=EXTREME_AR_LOW, color='red', linestyle='--', linewidth=2, label=f'Extreme low ({EXTREME_AR_LOW})')
ax1.axvline(x=EXTREME_AR_HIGH, color='red', linestyle='--', linewidth=2, label=f'Extreme high ({EXTREME_AR_HIGH})')
ax1.set_xlabel('Aspect Ratio (W/H)', fontsize=12)
ax1.set_ylabel('Frequency', fontsize=12)
ax1.set_title('Aspect Ratio Distribution by Category', fontsize=13, fontweight='bold')
ax1.legend(loc='upper right', fontsize=8)
ax1.set_xlim(0, 10)

# Box plot by category
ax2 = axes[1]
df_anns_ar_plot = df_anns[df_anns['aspect_ratio'].notna()].copy()
df_anns_ar_plot['aspect_ratio_clipped'] = df_anns_ar_plot['aspect_ratio'].clip(0, 10)

order = list(CAT_ID_TO_NAME.values())
palette = [CATEGORY_COLORS.get(CAT_NAME_TO_ID.get(name, 1), '#333') for name in order]

sns.boxplot(data=df_anns_ar_plot, x='category_name', y='aspect_ratio_clipped', 
            order=order, palette=palette, ax=ax2)
ax2.axhline(y=EXTREME_AR_LOW, color='red', linestyle='--', linewidth=2)
ax2.axhline(y=EXTREME_AR_HIGH, color='red', linestyle='--', linewidth=2)
ax2.set_xlabel('Category', fontsize=12)
ax2.set_ylabel('Aspect Ratio (W/H)', fontsize=12)
ax2.set_title('Aspect Ratio Box Plot by Category', fontsize=13, fontweight='bold')
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# -------------------- Flag Extreme Aspect Ratio (TABLES ONLY) --------------------
df_extreme_ar = df_anns[(df_anns['category_id'] == table_cat_id) & 
                        ((df_anns['aspect_ratio'] < EXTREME_AR_LOW) | 
                         (df_anns['aspect_ratio'] > EXTREME_AR_HIGH))].copy()

print(f"\n  EXTREME ASPECT RATIO TABLES (AR<{EXTREME_AR_LOW} or AR>{EXTREME_AR_HIGH}):")
print(f"   Found: {len(df_extreme_ar)} annotations")
if len(df_extreme_ar) > 0:
    print("   Sample flagged files:")
    for _, row in df_extreme_ar.head(5).iterrows():
        print(f"      - {row['file_name']} (ann_id: {row['id']}, AR: {row['aspect_ratio']:.3f})")

# -------------------- Input Resolution Recommendation --------------------
table_ar = df_anns[df_anns['category_id'] == table_cat_id]['aspect_ratio']
table_ar_median = table_ar.median()
table_ar_p25 = table_ar.quantile(0.25)
table_ar_p75 = table_ar.quantile(0.75)

print(f"\n TABLE Aspect Ratio Insights:")
print(f"   Median AR: {table_ar_median:.2f}")
print(f"   IQR: [{table_ar_p25:.2f}, {table_ar_p75:.2f}]")

if table_ar_median > 2.0:
    print(f"     Tables are predominantly WIDE. Consider wider input resolution (e.g., 1280x640).")
elif table_ar_median < 0.5:
    print(f"     Tables are predominantly TALL. Consider taller input resolution (e.g., 640x1280).")
else:
    print(f"     Tables have balanced AR. Standard square input (640x640 or 1280x1280) should work well.")

<a id='heatmap'></a>
## 6. Spatial Distribution Heatmap

In [ ]:
# ============================================================
# CELL 5: Spatial Distribution Heatmap
# ============================================================

print("="*60)
print(" SPATIAL DISTRIBUTION HEATMAP")
print("="*60)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# -------------------- All Annotations Heatmap --------------------
ax1 = axes[0]
h1 = ax1.hist2d(df_anns['norm_cx'], df_anns['norm_cy'], bins=30, cmap='YlOrRd', cmin=1)
ax1.set_xlabel('Normalized X (0=left, 1=right)', fontsize=11)
ax1.set_ylabel('Normalized Y (0=top, 1=bottom)', fontsize=11)
ax1.set_title('All Annotations Center Distribution', fontsize=12, fontweight='bold')
ax1.invert_yaxis()  # Top of image at top of plot
ax1.set_xlim(0, 1)
ax1.set_ylim(1, 0)
plt.colorbar(h1[3], ax=ax1, label='Count')

# -------------------- Tables Only Heatmap --------------------
ax2 = axes[1]
df_tables = df_anns[df_anns['category_id'] == table_cat_id]
h2 = ax2.hist2d(df_tables['norm_cx'], df_tables['norm_cy'], bins=20, cmap='Blues', cmin=1)
ax2.set_xlabel('Normalized X', fontsize=11)
ax2.set_ylabel('Normalized Y', fontsize=11)
ax2.set_title('TABLE Annotations Only', fontsize=12, fontweight='bold')
ax2.invert_yaxis()
ax2.set_xlim(0, 1)
ax2.set_ylim(1, 0)
plt.colorbar(h2[3], ax=ax2, label='Count')

# -------------------- Sub-components (non-table) Heatmap --------------------
ax3 = axes[2]
df_subcomponents = df_anns[df_anns['category_id'] != table_cat_id]
h3 = ax3.hist2d(df_subcomponents['norm_cx'], df_subcomponents['norm_cy'], bins=30, cmap='Greens', cmin=1)
ax3.set_xlabel('Normalized X', fontsize=11)
ax3.set_ylabel('Normalized Y', fontsize=11)
ax3.set_title('Sub-components (cells/rows/cols)', fontsize=12, fontweight='bold')
ax3.invert_yaxis()
ax3.set_xlim(0, 1)
ax3.set_ylim(1, 0)
plt.colorbar(h3[3], ax=ax3, label='Count')

plt.tight_layout()
plt.show()

# -------------------- Edge/Margin Analysis --------------------
margin_thresh = 0.05  # 5% from edge

# Check if annotations cluster near edges (possible margin annotations)
near_left = df_anns[df_anns['norm_cx'] < margin_thresh]
near_right = df_anns[df_anns['norm_cx'] > (1 - margin_thresh)]
near_top = df_anns[df_anns['norm_cy'] < margin_thresh]
near_bottom = df_anns[df_anns['norm_cy'] > (1 - margin_thresh)]

print(f"\n Annotations near edges (<{margin_thresh*100}% from edge):")
print(f"   Near left edge: {len(near_left)} ({len(near_left)/len(df_anns)*100:.1f}%)")
print(f"   Near right edge: {len(near_right)} ({len(near_right)/len(df_anns)*100:.1f}%)")
print(f"   Near top edge: {len(near_top)} ({len(near_top)/len(df_anns)*100:.1f}%)")
print(f"   Near bottom edge: {len(near_bottom)} ({len(near_bottom)/len(df_anns)*100:.1f}%)")

print("\n Interpretation: Most annotations should cluster toward the center. ")
print("   High edge counts might indicate margin/border annotations.")

<a id='hierarchy'></a>
## 7. Hierarchical Consistency Check

In [ ]:
# ============================================================
# CELL 6: Hierarchical Consistency Check
# ============================================================

print("="*60)
print(" HIERARCHICAL CONSISTENCY CHECK")
print("="*60)

# Get image IDs with different category types
images_with_tables = set(df_anns[df_anns['category_id'] == table_cat_id]['image_id'].unique())
images_with_children = set(df_anns[df_anns['category_id'] != table_cat_id]['image_id'].unique())

# -------------------- Check 1: Orphaned Children --------------------
# Images that have child annotations (cells, rows, cols) but no table parent
orphaned_children_images = images_with_children - images_with_tables

print(f"\n Check 1: Orphaned Child Annotations")
print(f"   Images with children but no table parent: {len(orphaned_children_images)}")

if len(orphaned_children_images) > 0:
    print("     This is a data quality issue - child annotations exist without parent table!")
    orphan_image_ids = list(orphaned_children_images)[:5]
    print(f"   Sample image IDs: {orphan_image_ids}")
    
    df_orphan_anns = df_anns[df_anns['image_id'].isin(orphaned_children_images)].copy()
else:
    print("    All child annotations have a parent table.")
    df_orphan_anns = pd.DataFrame()

# -------------------- Check 2: Tables without Children --------------------
# Images that have a table but zero child annotations (suspicious incomplete labeling)
tables_without_children = images_with_tables - images_with_children

print(f"\n Check 2: Tables without Child Annotations")
print(f"   Images with table but no children: {len(tables_without_children)}")

if len(tables_without_children) > 0:
    print("     These tables might be incompletely annotated!")
    sample_ids = list(tables_without_children)[:5]
    for img_id in sample_ids:
        fname = df_images[df_images['image_id'] == img_id]['file_name'].values[0]
        print(f"      - {fname}")
else:
    print("    All tables have child annotations.")

# -------------------- Check 3: Spatial Containment --------------------
# Verify child annotations are spatially within the parent table bbox

def check_containment(table_bbox, child_bbox, tolerance=10):
    """Check if child bbox is contained within table bbox (with tolerance in pixels)"""
    tx, ty, tw, th = table_bbox
    cx, cy, cw, ch = child_bbox
    
    # Child must be within table bounds (with some tolerance for annotation noise)
    return (cx >= tx - tolerance and 
            cy >= ty - tolerance and
            cx + cw <= tx + tw + tolerance and
            cy + ch <= ty + th + tolerance)

print(f"\n Check 3: Spatial Containment of Children within Tables")

containment_issues = []

# Group annotations by image
for image_id in images_with_tables:
    img_anns = df_anns[df_anns['image_id'] == image_id]
    
    # Get all table bboxes for this image
    table_bboxes = img_anns[img_anns['category_id'] == table_cat_id]['bbox'].tolist()
    
    # Get all child annotations
    children = img_anns[img_anns['category_id'] != table_cat_id]
    
    for _, child in children.iterrows():
        child_bbox = child['bbox']
        
        # Check if child is contained in ANY table bbox
        is_contained = any(check_containment(tb, child_bbox) for tb in table_bboxes)
        
        if not is_contained:
            containment_issues.append({
                'image_id': image_id,
                'file_name': child['file_name'],
                'ann_id': child['id'],
                'category': child['category_name'],
                'child_bbox': child_bbox
            })

df_containment_issues = pd.DataFrame(containment_issues)

print(f"   Child annotations outside parent table: {len(df_containment_issues)}")
if len(df_containment_issues) > 0:
    print("     These annotations are spatially outside their expected parent!")
    print("   Sample issues:")
    for _, row in df_containment_issues.head(5).iterrows():
        print(f"      - {row['file_name']} (ann_id: {row['ann_id']}, cat: {row['category']})")
else:
    print("    All child annotations are spatially within their parent tables.")

<a id='iou'></a>
## 8. Overlap / IoU Analysis

In [ ]:
# ============================================================
# CELL 7: Overlap / IoU Analysis (Duplicate Detection)
# ============================================================

print("="*60)
print(" IoU ANALYSIS: Duplicate Annotation Detection")
print("="*60)

def compute_iou(bbox1, bbox2):
    """Compute IoU between two bboxes in [x, y, w, h] format"""
    x1, y1, w1, h1 = bbox1
    x2, y2, w2, h2 = bbox2
    
    # Convert to [x1, y1, x2, y2] format
    box1 = [x1, y1, x1 + w1, y1 + h1]
    box2 = [x2, y2, x2 + w2, y2 + h2]
    
    # Compute intersection
    inter_x1 = max(box1[0], box2[0])
    inter_y1 = max(box1[1], box2[1])
    inter_x2 = min(box1[2], box2[2])
    inter_y2 = min(box1[3], box2[3])
    
    inter_w = max(0, inter_x2 - inter_x1)
    inter_h = max(0, inter_y2 - inter_y1)
    inter_area = inter_w * inter_h
    
    # Compute union
    area1 = w1 * h1
    area2 = w2 * h2
    union_area = area1 + area2 - inter_area
    
    if union_area == 0:
        return 0.0
    
    return inter_area / union_area

# Detect potential duplicates (same category, same image, high IoU)
print(f"\n Checking for duplicate annotations (IoU > {IOU_DUPLICATE_THRESH})...")

duplicates = []

# Group by image and category
for (image_id, cat_id), group in df_anns.groupby(['image_id', 'category_id']):
    if len(group) < 2:
        continue
    
    # Check all pairs within this group
    bboxes = group['bbox'].tolist()
    ann_ids = group['id'].tolist()
    
    for i, j in combinations(range(len(bboxes)), 2):
        iou = compute_iou(bboxes[i], bboxes[j])
        if iou > IOU_DUPLICATE_THRESH:
            fname = group['file_name'].iloc[0]
            cat_name = CAT_ID_TO_NAME.get(cat_id, 'unknown')
            duplicates.append({
                'image_id': image_id,
                'file_name': fname,
                'category': cat_name,
                'ann_id_1': ann_ids[i],
                'ann_id_2': ann_ids[j],
                'iou': iou
            })

df_duplicates = pd.DataFrame(duplicates)

print(f"\n  POTENTIAL DUPLICATE ANNOTATIONS (IoU > {IOU_DUPLICATE_THRESH}):")
print(f"   Found: {len(df_duplicates)} pairs")

if len(df_duplicates) > 0:
    print("\n   Breakdown by category:")
    dup_by_cat = df_duplicates['category'].value_counts()
    for cat, count in dup_by_cat.items():
        print(f"      {cat}: {count} pairs")
    
    print("\n   Sample duplicates:")
    for _, row in df_duplicates.head(5).iterrows():
        print(f"      - {row['file_name']} | {row['category']} | IDs: {row['ann_id_1']}, {row['ann_id_2']} | IoU: {row['iou']:.3f}")
else:
    print("    No duplicate annotations detected.")

# -------------------- IoU Distribution Visualization --------------------
# Sample IoU values for visualization (all pairs within same image/category)
print("\n Computing IoU distribution for same-category pairs (sampled)...")

sample_ious = []
max_samples = 5000

for (image_id, cat_id), group in df_anns.groupby(['image_id', 'category_id']):
    if len(group) < 2:
        continue
    
    bboxes = group['bbox'].tolist()
    
    # Sample pairs if too many
    pairs = list(combinations(range(len(bboxes)), 2))
    if len(pairs) > 10:
        pairs = pairs[:10]
    
    for i, j in pairs:
        iou = compute_iou(bboxes[i], bboxes[j])
        sample_ious.append({'iou': iou, 'category': CAT_ID_TO_NAME.get(cat_id, 'unknown')})
        
        if len(sample_ious) >= max_samples:
            break
    
    if len(sample_ious) >= max_samples:
        break

if len(sample_ious) > 0:
    df_ious = pd.DataFrame(sample_ious)
    
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.hist(df_ious['iou'], bins=50, color='#3498DB', edgecolor='black', alpha=0.7)
    ax.axvline(x=IOU_DUPLICATE_THRESH, color='red', linestyle='--', linewidth=2, 
               label=f'Duplicate threshold ({IOU_DUPLICATE_THRESH})')
    ax.set_xlabel('IoU', fontsize=12)
    ax.set_ylabel('Frequency', fontsize=12)
    ax.set_title('IoU Distribution (Same-Category Pairs, Same Image)', fontsize=13, fontweight='bold')
    ax.legend()
    plt.tight_layout()
    plt.show()
    
    print(f"\n   Sampled {len(df_ious)} IoU values.")
    print(f"   Mean IoU: {df_ious['iou'].mean():.3f}")
    print(f"   IoU > {IOU_DUPLICATE_THRESH}: {(df_ious['iou'] > IOU_DUPLICATE_THRESH).sum()} samples")

<a id='negatives'></a>
## 9. Negative Sample Strategy (Core Requirement 2)

In [ ]:
# ============================================================
# CELL 8: Negative Sample Strategy
# ============================================================

print("="*60)
print(" NEGATIVE SAMPLE STRATEGY")
print("="*60)

# Identify images with annotations
images_with_annotations = set(df_anns['image_id'].unique())
all_image_ids = set(df_images['image_id'].unique())

# Empty images (no annotations)
empty_image_ids = all_image_ids - images_with_annotations
df_empty = df_images[df_images['image_id'].isin(empty_image_ids)].copy()

# Calculate ratio
total_images = len(df_images)
n_empty = len(df_empty)
n_annotated = len(images_with_annotations)
empty_ratio = n_empty / total_images

print(f"\n Current Dataset Composition:")
print(f"   Total images: {total_images:,}")
print(f"   Annotated images: {n_annotated:,} ({n_annotated/total_images*100:.1f}%)")
print(f"   Empty images (negatives): {n_empty:,} ({empty_ratio*100:.1f}%)")

# -------------------- Visualization --------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Pie chart
ax1 = axes[0]
sizes = [n_annotated, n_empty]
labels = [f'Annotated\n({n_annotated:,})', f'Empty/Negative\n({n_empty:,})']
colors = ['#3498DB', '#E74C3C']
explode = (0, 0.05)
ax1.pie(sizes, explode=explode, labels=labels, colors=colors, autopct='%1.1f%%',
        shadow=True, startangle=90)
ax1.set_title('Dataset Composition', fontsize=13, fontweight='bold')

# Bar chart with target range
ax2 = axes[1]
categories = ['Current', 'Target Min', 'Target Max']
values = [empty_ratio * 100, NEGATIVE_RATIO_MIN * 100, NEGATIVE_RATIO_MAX * 100]
colors = ['#3498DB', '#2ECC71', '#2ECC71']

bars = ax2.bar(categories, values, color=colors, edgecolor='black', linewidth=1)
ax2.axhline(y=NEGATIVE_RATIO_MIN * 100, color='green', linestyle='--', linewidth=2, alpha=0.7)
ax2.axhline(y=NEGATIVE_RATIO_MAX * 100, color='green', linestyle='--', linewidth=2, alpha=0.7)
ax2.fill_between([-0.5, 2.5], NEGATIVE_RATIO_MIN * 100, NEGATIVE_RATIO_MAX * 100, 
                  color='green', alpha=0.1)
ax2.set_ylabel('Percentage (%)', fontsize=12)
ax2.set_title('Negative Sample Ratio', fontsize=13, fontweight='bold')
ax2.set_ylim(0, max(20, empty_ratio * 100 + 5))

for bar, val in zip(bars, values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
             f'{val:.1f}%', ha='center', va='bottom', fontsize=11)

plt.tight_layout()
plt.show()

# -------------------- Recommendation --------------------
print(f"\n RECOMMENDATION:")

if empty_ratio < NEGATIVE_RATIO_MIN:
    needed = int(NEGATIVE_RATIO_MIN * n_annotated / (1 - NEGATIVE_RATIO_MIN)) - n_empty
    print(f"    Current negative ratio ({empty_ratio*100:.1f}%) is BELOW target range ({NEGATIVE_RATIO_MIN*100}%-{NEGATIVE_RATIO_MAX*100}%).")
    print(f"     ADD approximately {needed:,} more empty images.")
    print(f"     Prioritize:")
    print(f"      - Pages with Bibliographies (dense, structured text)")
    print(f"      - Pages with Double-Column Text (visual similarity to tables)")
    print(f"      - Document pages with lists or structured content (confusable with tables)")
elif empty_ratio > NEGATIVE_RATIO_MAX:
    excess = n_empty - int(NEGATIVE_RATIO_MAX * n_annotated / (1 - NEGATIVE_RATIO_MAX))
    print(f"    Current negative ratio ({empty_ratio*100:.1f}%) is ABOVE target range ({NEGATIVE_RATIO_MIN*100}%-{NEGATIVE_RATIO_MAX*100}%).")
    print(f"     REMOVE approximately {excess:,} empty images, OR add more annotated samples.")
else:
    print(f"    Current negative ratio ({empty_ratio*100:.1f}%) is within target range ({NEGATIVE_RATIO_MIN*100}%-{NEGATIVE_RATIO_MAX*100}%).")
    print(f"     Ensure empty images include hard negatives (bibliographies, multi-column layouts).")

# -------------------- Display Sample Empty Images --------------------
if n_empty > 0:
    print(f"\n  Sample Empty Images ({min(9, n_empty)} shown):")
    print("   (Visually verify these include hard negatives like bibliographies/multi-column text)")
    
    sample_empty = df_empty.sample(min(9, n_empty), random_state=42)
    
    n_samples = len(sample_empty)
    n_cols = min(3, n_samples)
    n_rows = (n_samples + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4*n_cols, 4*n_rows))
    if n_samples == 1:
        axes = np.array([[axes]])
    elif n_rows == 1:
        axes = axes.reshape(1, -1)
    
    for idx, (_, row) in enumerate(sample_empty.iterrows()):
        r, c = idx // n_cols, idx % n_cols
        ax = axes[r, c]
        
        img_path = os.path.join(IMAGES_DIR, row['file_name'])
        if os.path.exists(img_path):
            img = Image.open(img_path)
            ax.imshow(img, cmap='gray')
            ax.set_title(row['file_name'], fontsize=9)
        else:
            ax.text(0.5, 0.5, f"Image not found:\n{row['file_name']}", 
                   ha='center', va='center', transform=ax.transAxes)
        ax.axis('off')
    
    # Hide empty subplots
    for idx in range(n_samples, n_rows * n_cols):
        r, c = idx // n_cols, idx % n_cols
        axes[r, c].axis('off')
    
    plt.tight_layout()
    plt.show()
else:
    print("\n   ⚠️  No empty images found in the dataset!")

<a id='outlier-viz'></a>
## 10. Outlier Sample Visualization

In [ ]:
# ============================================================
# CELL 9: Outlier Sample Visualization
# ============================================================

print("="*60)
print(" OUTLIER SAMPLE VISUALIZATION")
print("="*60)

def visualize_annotations(df_subset, title, max_samples=4):
    """Visualize sample images with their annotations."""
    if len(df_subset) == 0:
        print(f"\n   No samples to visualize for: {title}")
        return
    
    # Get unique images
    unique_images = df_subset['image_id'].unique()[:max_samples]
    n_samples = len(unique_images)
    
    if n_samples == 0:
        return
    
    n_cols = min(2, n_samples)
    n_rows = (n_samples + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(6*n_cols, 6*n_rows))
    if n_samples == 1:
        axes = np.array([[axes]])
    elif n_rows == 1:
        axes = axes.reshape(1, -1)
    
    for idx, img_id in enumerate(unique_images):
        r, c = idx // n_cols, idx % n_cols
        ax = axes[r, c]
        
        img_row = df_images[df_images['image_id'] == img_id].iloc[0]
        img_path = os.path.join(IMAGES_DIR, img_row['file_name'])
        
        if os.path.exists(img_path):
            img = Image.open(img_path)
            ax.imshow(img, cmap='gray')
            
            # Draw all annotations for this image that are in the subset
            img_anns = df_subset[df_subset['image_id'] == img_id]
            
            for _, ann in img_anns.iterrows():
                x, y, w, h = ann['bbox']
                cat_id = ann['category_id']
                color = CATEGORY_COLORS.get(cat_id, 'red')
                
                rect = Rectangle((x, y), w, h, linewidth=2, 
                                 edgecolor=color, facecolor='none')
                ax.add_patch(rect)
                
                # Add label
                ax.text(x, y-5, f"{ann['category_name'][:10]}", 
                       color=color, fontsize=8, fontweight='bold',
                       bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7))
            
            ax.set_title(f"{img_row['file_name']}\n({len(img_anns)} flagged annotations)", fontsize=10)
        else:
            ax.text(0.5, 0.5, f"Image not found:\n{img_row['file_name']}", 
                   ha='center', va='center', transform=ax.transAxes)
        ax.axis('off')
    
    # Hide empty subplots
    for idx in range(n_samples, n_rows * n_cols):
        r, c = idx // n_cols, idx % n_cols
        axes[r, c].axis('off')
    
    fig.suptitle(title, fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

# Create legend
print("\n Category Color Legend:")
for cat_id, cat_name in CAT_ID_TO_NAME.items():
    color = CATEGORY_COLORS.get(cat_id, '#333')
    print(f"   {cat_name}: {color}")

# -------------------- Visualize Each Flagged Category --------------------

# 1. Tiny table boxes
print(f"\n" + "="*50)
print(f" TINY TABLE BOXES (<{TINY_AREA_PCT*100}% of page)")
print("="*50)
visualize_annotations(df_tiny_tables, f"Tiny Table Boxes (<{TINY_AREA_PCT*100}% of page)")

# 2. Full-page boxes
print(f"\n" + "="*50)
print(f" FULL-PAGE BOXES (>{FULL_PAGE_PCT*100}% of page)")
print("="*50)
visualize_annotations(df_fullpage, f"Full-Page Boxes (>{FULL_PAGE_PCT*100}% of page)")

# 3. Extreme aspect ratio tables
print(f"\n" + "="*50)
print(f" EXTREME ASPECT RATIO TABLES")
print("="*50)
visualize_annotations(df_extreme_ar, "Extreme Aspect Ratio Tables")

# 4. Orphaned children (if any)
if len(df_orphan_anns) > 0:
    print(f"\n" + "="*50)
    print(f" ORPHANED CHILD ANNOTATIONS")
    print("="*50)
    visualize_annotations(df_orphan_anns, "Orphaned Child Annotations (no parent table)")

# 5. Containment issues (if any)
if len(df_containment_issues) > 0:
    print(f"\n" + "="*50)
    print(f" CONTAINMENT ISSUES (children outside parent)")
    print("="*50)
    # Get the annotation rows for visualization
    containment_ann_ids = df_containment_issues['ann_id'].tolist()
    df_containment_viz = df_anns[df_anns['id'].isin(containment_ann_ids)]
    visualize_annotations(df_containment_viz, "Children Outside Parent Table")

<a id='summary'></a>
## 11. Summary Report & Recommendations

In [ ]:
# ============================================================
# CELL 10: Summary Report & Recommendations
# ============================================================

print("="*70)
print(" PHASE 1 EDA SUMMARY REPORT")
print("="*70)

# -------------------- Compile All Flagged Annotations --------------------
flagged_records = []

# Tiny tables
for _, row in df_tiny_tables.iterrows():
    flagged_records.append({
        'ann_id': row['id'],
        'image_id': row['image_id'],
        'file_name': row['file_name'],
        'category': row['category_name'],
        'issue_type': 'tiny_box',
        'details': f"area_pct={row['box_area_pct']*100:.2f}%"
    })

# Full-page boxes
for _, row in df_fullpage.iterrows():
    flagged_records.append({
        'ann_id': row['id'],
        'image_id': row['image_id'],
        'file_name': row['file_name'],
        'category': row['category_name'],
        'issue_type': 'fullpage_box',
        'details': f"area_pct={row['box_area_pct']*100:.2f}%"
    })

# Extreme AR tables
for _, row in df_extreme_ar.iterrows():
    flagged_records.append({
        'ann_id': row['id'],
        'image_id': row['image_id'],
        'file_name': row['file_name'],
        'category': row['category_name'],
        'issue_type': 'extreme_aspect_ratio',
        'details': f"AR={row['aspect_ratio']:.3f}"
    })

# Orphaned children
if len(df_orphan_anns) > 0:
    for _, row in df_orphan_anns.iterrows():
        flagged_records.append({
            'ann_id': row['id'],
            'image_id': row['image_id'],
            'file_name': row['file_name'],
            'category': row['category_name'],
            'issue_type': 'orphaned_child',
            'details': 'no parent table'
        })

# Containment issues
for _, row in df_containment_issues.iterrows():
    flagged_records.append({
        'ann_id': row['ann_id'],
        'image_id': row['image_id'],
        'file_name': row['file_name'],
        'category': row['category'],
        'issue_type': 'outside_parent',
        'details': 'child outside parent bbox'
    })

# Duplicates (flag one of each pair)
for _, row in df_duplicates.iterrows():
    flagged_records.append({
        'ann_id': row['ann_id_2'],  # Flag the second one for removal
        'image_id': row['image_id'],
        'file_name': row['file_name'],
        'category': row['category'],
        'issue_type': 'duplicate',
        'details': f"IoU={row['iou']:.3f} with ann_id={row['ann_id_1']}"
    })

df_flagged = pd.DataFrame(flagged_records)

# -------------------- Summary Table --------------------
print("\n" + "-"*70)
print(" ISSUE SUMMARY")
print("-"*70)

summary_data = [
    ['Tiny table boxes (<3%)', len(df_tiny_tables), 'Review/remove - likely mislabeled cells'],
    ['Full-page boxes (>95%)', len(df_fullpage), 'Review/remove - likely margin annotations'],
    ['Extreme AR tables', len(df_extreme_ar), 'Adjust input resolution or review'],
    ['Orphaned child annotations', len(df_orphan_anns) if len(df_orphan_anns) > 0 else 0, 'Fix hierarchy - add parent table'],
    ['Children outside parent', len(df_containment_issues), 'Review spatial alignment'],
    ['Duplicate annotations', len(df_duplicates), 'De-duplicate'],
    ['Tables without children', len(tables_without_children), 'Complete annotations if needed'],
]

df_summary = pd.DataFrame(summary_data, columns=['Issue Type', 'Count', 'Recommended Action'])
display(df_summary)

# -------------------- Negative Sample Status --------------------
print("\n" + "-"*70)
print(" NEGATIVE SAMPLE STATUS")
print("-"*70)

print(f"   Current ratio: {empty_ratio*100:.1f}%")
print(f"   Target range: {NEGATIVE_RATIO_MIN*100}% - {NEGATIVE_RATIO_MAX*100}%")

if empty_ratio < NEGATIVE_RATIO_MIN:
    needed = int(NEGATIVE_RATIO_MIN * n_annotated / (1 - NEGATIVE_RATIO_MIN)) - n_empty
    print(f"   Status: BELOW TARGET - Add ~{needed:,} empty images")
elif empty_ratio > NEGATIVE_RATIO_MAX:
    excess = n_empty - int(NEGATIVE_RATIO_MAX * n_annotated / (1 - NEGATIVE_RATIO_MAX))
    print(f"   Status: ABOVE TARGET - Remove ~{excess:,} empty images")
else:
    print(f"   Status: WITHIN TARGET")

# -------------------- Input Resolution Recommendation --------------------
print("\n" + "-"*70)
print(" YOLO INPUT RESOLUTION RECOMMENDATION")
print("-"*70)

print(f"   Image mean AR: {mean_ar:.3f}")
print(f"   Table median AR: {table_ar_median:.3f}")

# Combined recommendation
if 0.6 <= table_ar_median <= 1.5:
    final_rec = "640x640 (balanced) or 1280x1280 (high-res)"
elif table_ar_median > 1.5:
    final_rec = "1280x640 or 960x640 (landscape, for wide tables)"
else:
    final_rec = "640x960 or 640x1280 (portrait, for tall tables)"

print(f"   Recommended: {final_rec}")

# -------------------- Export Flagged Annotations --------------------
output_csv_path = os.path.join(OUTPUT_DIR, 'flagged_annotations.csv')

if len(df_flagged) > 0:
    df_flagged.to_csv(output_csv_path, index=False)
    print(f"\n Exported {len(df_flagged)} flagged annotations to:")
    print(f"   {output_csv_path}")
    
    # Show issue type breakdown
    print("\n   Breakdown by issue type:")
    for issue_type, count in df_flagged['issue_type'].value_counts().items():
        print(f"      {issue_type}: {count}")
else:
    print(f"\n No annotations flagged for review!")

# -------------------- Final Checklist --------------------
print("\n" + "="*70)
print(" PHASE 1 COMPLETION CHECKLIST")
print("="*70)

checklist = [
    ("Area distribution analyzed", True),
    ("Tiny boxes flagged (tables only)", len(df_tiny_tables) >= 0),
    ("Full-page boxes flagged", len(df_fullpage) >= 0),
    ("Aspect ratio analyzed", True),
    ("Extreme AR tables flagged", len(df_extreme_ar) >= 0),
    ("Spatial heatmap generated", True),
    ("Hierarchical consistency checked", True),
    ("IoU duplicates checked", True),
    ("Negative sample ratio evaluated", True),
    ("Outlier samples visualized", True),
    ("Flagged annotations exported", len(df_flagged) >= 0),
]

for task, done in checklist:
    status = "OK" if done else "NOK"
    print(f"   {status} {task}")

print("\n" + "="*70)
print(" READY FOR PHASE 2: DATA CONVERSION")
print("="*70)
print("\n   Next steps:")
print("   1. Review flagged annotations in 'flagged_annotations.csv'")
print("   2. Clean dataset by removing/correcting problematic annotations")
print("   3. Add hard-negative empty images if needed")
print("   4. Proceed to Phase 2: COCO -> YOLO format conversion")

# Phase 2: Advanced Preprocessing & Data Engineering

**Objective:** Transform raw COCO annotations into a clean, YOLO-ready dataset optimized for YOLOv10 table detection.

---

## Pipeline Overview

| Step | Operation | Key Enhancement |
|------|-----------|----------------|
| 1 | Configuration & Setup | Kaggle paths, split ratios, seed |
| 2 | Load & Parse COCO JSON | Build DataFrames, category mapping |
| 3 | Semantic Label Filtering | Table-only (cat 1) + bbox expansion safety net |
| 4 | Coordinate Normalization | COCO → YOLO with clipping & sanitization |
| 5 | Document-Grouped Splitting | 85/10/5 split grouped by paper ID (no leakage) |
| 6 | YOLO Directory Export | Images + labels + data.yaml |
| 7 | Visual Verification | Overlay YOLO boxes on sample images |
| 8 | Summary Report | Statistics, integrity checks, completion checklist |

---

## Table of Contents

1. [Configuration & Setup](#config)
2. [Load & Parse COCO JSON](#load)
3. [Semantic Label Filtering](#filtering)
4. [Coordinate Normalization & Integrity](#normalization)
5. [Document-Grouped Dataset Splitting](#splitting)
6. [YOLO Directory Export](#export)
7. [Visual Verification](#verification)
8. [Summary Report & Completion Checklist](#summary)

---

<a id='config'></a>
## 1. Configuration & Setup

In [ ]:
import json
import os
import re
import shutil
import warnings
import textwrap
import time
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import seaborn as sns
from PIL import Image
from sklearn.model_selection import GroupShuffleSplit

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# =====================================================================
# PATHS  (Kaggle environment)
# =====================================================================
JSON_PATH   = '/kaggle/input/datasets/mohammedahmedxx12/machathon-dataset/Phase1_Train_Dataset/annotations/Cells_Anotations_coco.json'
IMAGES_DIR  = '/kaggle/input/datasets/mohammedahmedxx12/machathon-dataset/Phase1_Train_Dataset/images'
OUTPUT_ROOT = '/kaggle/working/yolo_dataset'   # output directory for the YOLO dataset

# =====================================================================
# DATASET SPLIT RATIOS
# =====================================================================
TRAIN_RATIO = 0.85
VAL_RATIO   = 0.10
TEST_RATIO  = 0.05
RANDOM_SEED = 42

assert abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1.0) < 1e-9, "Split ratios must sum to 1.0"

# =====================================================================
# YOLO CLASS MAPPING
# =====================================================================
# COCO category_id 1 (table) --> YOLO class_id 0
COCO_TABLE_CAT_ID = 1
YOLO_TABLE_CLASS   = 0
CLASS_NAMES        = ['table']

# =====================================================================
# EXPANSION SAFETY-NET PARAMS
# =====================================================================
# Category 5 = "table projected row header" - might sit outside parent table
EXPANDABLE_CAT_ID  = 5
CONTAINMENT_TOL_PX = 5   # tolerance in pixels for containment check

# =====================================================================
# DOCUMENT ID REGEX
# =====================================================================
DOC_ID_PATTERN = re.compile(r'^(PMC\d+)_\d+\.jpg$')

# =====================================================================
# VISUALIZATION
# =====================================================================
TABLE_COLOR = '#E74C3C'
sns.set_theme(style='whitegrid', palette='husl')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print("Phase 2 configuration loaded.")
print(f"   Train / Val / Test = {TRAIN_RATIO:.0%} / {VAL_RATIO:.0%} / {TEST_RATIO:.0%}")
print(f"   Output root: {OUTPUT_ROOT}")
print(f"   Random seed: {RANDOM_SEED}")

<a id='load'></a>
## 2. Load & Parse COCO JSON

In [ ]:
# ============================================================
# Load COCO JSON
# ============================================================
with open(JSON_PATH, 'r') as f:
    coco_data = json.load(f)

images_list      = coco_data['images']
annotations_list = coco_data['annotations']
categories_list  = coco_data['categories']

print(f"   Loaded {len(images_list):,} images")
print(f"   Loaded {len(annotations_list):,} annotations")
print(f"   Loaded {len(categories_list)} categories")

# -------------------- Category Mapping --------------------
df_cats = pd.DataFrame(categories_list)
CAT_ID_TO_NAME = dict(zip(df_cats['id'], df_cats['name']))
CAT_NAME_TO_ID = dict(zip(df_cats['name'], df_cats['id']))

print("\n Categories:")
for _, row in df_cats.iterrows():
    marker = " <-- TARGET" if row['id'] == COCO_TABLE_CAT_ID else "     DROP"
    print(f"   {row['id']}: {row['name']} (parent: {row['supercategory']}) {marker}")

# -------------------- Images DataFrame --------------------
df_images = pd.DataFrame(images_list)
df_images = df_images.rename(columns={'id': 'image_id'})

# Extract document ID from filename  (PMC1064076_6.jpg --> PMC1064076)
df_images['doc_id'] = df_images['file_name'].apply(
    lambda f: DOC_ID_PATTERN.match(f).group(1) if DOC_ID_PATTERN.match(f) else f
)

# -------------------- Annotations DataFrame --------------------
df_anns = pd.DataFrame(annotations_list)
df_anns['bbox_x'] = df_anns['bbox'].apply(lambda b: b[0])
df_anns['bbox_y'] = df_anns['bbox'].apply(lambda b: b[1])
df_anns['bbox_w'] = df_anns['bbox'].apply(lambda b: b[2])
df_anns['bbox_h'] = df_anns['bbox'].apply(lambda b: b[3])

# Merge image metadata
df_anns = df_anns.merge(
    df_images[['image_id', 'file_name', 'width', 'height', 'doc_id']],
    on='image_id', how='left'
)
df_anns['category_name'] = df_anns['category_id'].map(CAT_ID_TO_NAME)

print(f"\n DataFrames built successfully.")
print(f"   df_images shape: {df_images.shape}")
print(f"   df_anns shape:   {df_anns.shape}")
print(f"   Unique documents: {df_images['doc_id'].nunique()}")

# Quick annotation breakdown
print("\n Annotations by category:")
for cat_id, cat_name in CAT_ID_TO_NAME.items():
    count = (df_anns['category_id'] == cat_id).sum()
    print(f"   {cat_name}: {count:,}")

<a id='filtering'></a>
## 3. Semantic Label Filtering

**Goal:** Keep only `category_id == 1` (table) annotations. Before dropping children, check if any `table projected row header` (cat 5) boxes sit outside their parent table box. If so, expand the table box to encompass them — maximizing the containment area.

Phase 1 found 0 containment violations. This cell acts as a **safety net** for robustness.

In [ ]:
# ============================================================
# Step 3: Semantic Label Filtering + Table BBox Expansion
# ============================================================

print("="*60)
print(" SEMANTIC LABEL FILTERING")
print("="*60)

n_total_anns = len(df_anns)
n_table_anns_before = (df_anns['category_id'] == COCO_TABLE_CAT_ID).sum()
n_dropped = n_total_anns - n_table_anns_before

print(f"\n Before filtering:")
print(f"   Total annotations: {n_total_anns:,}")
print(f"   Table annotations (cat {COCO_TABLE_CAT_ID}): {n_table_anns_before:,}")
print(f"   Non-table annotations to drop: {n_dropped:,}")

# ============================================================
# SAFETY NET: Expand table boxes to encompass stray projected row headers
# ============================================================
print(f"\n Running table bbox expansion safety net...")
print(f"   Checking if any cat {EXPANDABLE_CAT_ID} (table projected row header) boxes")
print(f"   fall outside their parent table bboxes (tolerance: {CONTAINMENT_TOL_PX}px)...")

expansion_count = 0
expansion_log = []

# Work on a copy of table annotations to allow in-place bbox modification
df_tables_work = df_anns[df_anns['category_id'] == COCO_TABLE_CAT_ID].copy()
df_projected = df_anns[df_anns['category_id'] == EXPANDABLE_CAT_ID].copy()

# Build a lookup: image_id -> list of table annotation indices (in df_tables_work)
table_idx_by_image = defaultdict(list)
for idx, row in df_tables_work.iterrows():
    table_idx_by_image[row['image_id']].append(idx)

for _, proj_row in df_projected.iterrows():
    img_id = proj_row['image_id']
    px, py, pw, ph = proj_row['bbox_x'], proj_row['bbox_y'], proj_row['bbox_w'], proj_row['bbox_h']
    p_x2 = px + pw
    p_y2 = py + ph
    
    table_indices = table_idx_by_image.get(img_id, [])
    if not table_indices:
        continue  # No table on this image (shouldn't happen per Phase 1)
    
    # Check containment in each table box
    contained = False
    best_table_idx = None
    best_overlap = -1
    
    for t_idx in table_indices:
        t = df_tables_work.loc[t_idx]
        tx, ty, tw, th = t['bbox_x'], t['bbox_y'], t['bbox_w'], t['bbox_h']
        t_x2 = tx + tw
        t_y2 = ty + th
        
        # Containment check with tolerance
        if (px >= tx - CONTAINMENT_TOL_PX and
            py >= ty - CONTAINMENT_TOL_PX and
            p_x2 <= t_x2 + CONTAINMENT_TOL_PX and
            p_y2 <= t_y2 + CONTAINMENT_TOL_PX):
            contained = True
            break
        
        # Track the table with max overlap (for expansion target)
        inter_x1 = max(px, tx)
        inter_y1 = max(py, ty)
        inter_x2 = min(p_x2, t_x2)
        inter_y2 = min(p_y2, t_y2)
        overlap = max(0, inter_x2 - inter_x1) * max(0, inter_y2 - inter_y1)
        
        if overlap > best_overlap:
            best_overlap = overlap
            best_table_idx = t_idx
    
    if not contained and best_table_idx is not None:
        # Expand the nearest table box to encompass this projected row header
        t = df_tables_work.loc[best_table_idx]
        old_tx, old_ty = t['bbox_x'], t['bbox_y']
        old_tx2 = old_tx + t['bbox_w']
        old_ty2 = old_ty + t['bbox_h']
        
        new_tx  = min(old_tx,  px)
        new_ty  = min(old_ty,  py)
        new_tx2 = max(old_tx2, p_x2)
        new_ty2 = max(old_ty2, p_y2)
        
        # Clamp to image boundaries
        img_w = proj_row['width']
        img_h = proj_row['height']
        new_tx  = max(0, new_tx)
        new_ty  = max(0, new_ty)
        new_tx2 = min(img_w, new_tx2)
        new_ty2 = min(img_h, new_ty2)
        
        new_tw = new_tx2 - new_tx
        new_th = new_ty2 - new_ty
        
        # Update the table bbox in-place
        df_tables_work.at[best_table_idx, 'bbox_x'] = new_tx
        df_tables_work.at[best_table_idx, 'bbox_y'] = new_ty
        df_tables_work.at[best_table_idx, 'bbox_w'] = new_tw
        df_tables_work.at[best_table_idx, 'bbox_h'] = new_th
        df_tables_work.at[best_table_idx, 'bbox'] = [new_tx, new_ty, new_tw, new_th]
        
        expansion_count += 1
        expansion_log.append({
            'image_id': img_id,
            'file_name': proj_row['file_name'],
            'table_ann_id': int(t['id']),
            'proj_ann_id': int(proj_row['id']),
            'old_bbox': [old_tx, old_ty, t['bbox_w'], t['bbox_h']],
            'new_bbox': [new_tx, new_ty, new_tw, new_th]
        })

# Report expansion results
if expansion_count > 0:
    print(f"\n   Table boxes expanded: {expansion_count}")
    print(f"   Expansion log (first 5):")
    for entry in expansion_log[:5]:
        print(f"      {entry['file_name']}: table ann {entry['table_ann_id']}")
        print(f"         old: {[round(v,1) for v in entry['old_bbox']]}")
        print(f"         new: {[round(v,1) for v in entry['new_bbox']]}")
else:
    print(f"\n   No expansions needed. All projected row headers are within parent tables.")

# ============================================================
# Final filtered DataFrame: table-only
# ============================================================
df_tables = df_tables_work.copy()

print(f"\n After filtering:")
print(f"   Table annotations retained: {len(df_tables):,}")
print(f"   Annotations dropped: {n_dropped:,} ({n_dropped/n_total_anns*100:.1f}%)")
print(f"   Table boxes expanded: {expansion_count}")

# How many images have at least one table?
images_with_tables = set(df_tables['image_id'].unique())
images_without_tables = set(df_images['image_id'].unique()) - images_with_tables
print(f"\n   Images WITH table annotations: {len(images_with_tables):,}")
print(f"   Images WITHOUT table annotations: {len(images_without_tables):,}")
print(f"   (Images without tables will get empty label files)")

<a id='normalization'></a>
## 4. Coordinate Normalization & Integrity

Convert COCO `[x, y, w, h]` pixel coordinates to YOLO normalized format:

$$\text{x\_center} = \frac{x_{\min} + x_{\max}}{2 \cdot W}, \quad \text{y\_center} = \frac{y_{\min} + y_{\max}}{2 \cdot H}$$

$$\text{width} = \frac{x_{\max} - x_{\min}}{W}, \quad \text{height} = \frac{y_{\max} - y_{\min}}{H}$$

**Sanitization:**
- Clip all edge coordinates to `[0, 1]` **before** computing center/size (geometrically correct)
- Discard boxes with `width <= 0` or `height <= 0` after clipping

In [ ]:
# ============================================================
# Step 4: COCO -> YOLO Coordinate Conversion with Sanitization
# ============================================================

print("="*60)
print(" COORDINATE NORMALIZATION & INTEGRITY")
print("="*60)

n_before = len(df_tables)
n_clipped = 0

yolo_records = []  # Each entry: (image_id, file_name, yolo_line_str)

for _, row in df_tables.iterrows():
    img_w = row['width']
    img_h = row['height']
    bx, by, bw, bh = row['bbox_x'], row['bbox_y'], row['bbox_w'], row['bbox_h']
    
    # Compute raw normalized edge coordinates
    x_min_raw = bx / img_w
    y_min_raw = by / img_h
    x_max_raw = (bx + bw) / img_w
    y_max_raw = (by + bh) / img_h
    
    # Clip edges to [0, 1]
    x_min = np.clip(x_min_raw, 0.0, 1.0)
    y_min = np.clip(y_min_raw, 0.0, 1.0)
    x_max = np.clip(x_max_raw, 0.0, 1.0)
    y_max = np.clip(y_max_raw, 0.0, 1.0)
    
    # Track if clipping occurred
    if (x_min != x_min_raw or y_min != y_min_raw or
        x_max != x_max_raw or y_max != y_max_raw):
        n_clipped += 1
    
    # Derive YOLO center + size from clipped edges
    w_norm = x_max - x_min
    h_norm = y_max - y_min
    
    # Discard degenerate boxes
    if w_norm <= 0 or h_norm <= 0:
        continue
    
    x_center = (x_min + x_max) / 2.0
    y_center = (y_min + y_max) / 2.0
    
    yolo_line = f"{YOLO_TABLE_CLASS} {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}"
    
    yolo_records.append({
        'image_id': row['image_id'],
        'file_name': row['file_name'],
        'ann_id': row['id'],
        'yolo_line': yolo_line,
        'x_center': x_center,
        'y_center': y_center,
        'w_norm': w_norm,
        'h_norm': h_norm
    })

df_yolo = pd.DataFrame(yolo_records)
n_after = len(df_yolo)
n_discarded = n_before - n_after

print(f"\n Conversion Results:")
print(f"   Input table annotations: {n_before:,}")
print(f"   Boxes clipped to [0,1]:  {n_clipped}")
print(f"   Boxes discarded (w<=0 or h<=0): {n_discarded}")
print(f"   Valid YOLO labels: {n_after:,}")

# -------------------- Sanity Checks --------------------
print(f"\n Coordinate Range Verification:")
for col in ['x_center', 'y_center', 'w_norm', 'h_norm']:
    vmin = df_yolo[col].min()
    vmax = df_yolo[col].max()
    status = "OK" if 0.0 <= vmin and vmax <= 1.0 else "FAIL"
    print(f"   {col:>10s}: [{vmin:.6f}, {vmax:.6f}]  {status}")

# -------------------- Distribution Visualization --------------------
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))

for ax, col, label in zip(axes, 
                          ['x_center', 'y_center', 'w_norm', 'h_norm'],
                          ['X Center', 'Y Center', 'Width', 'Height']):
    ax.hist(df_yolo[col], bins=40, color='#3498DB', edgecolor='black', alpha=0.7)
    ax.set_xlabel(label, fontsize=11)
    ax.set_ylabel('Count', fontsize=11)
    ax.set_title(f'YOLO {label} Distribution', fontsize=11, fontweight='bold')
    ax.set_xlim(0, 1)

plt.suptitle('Normalized YOLO Coordinate Distributions', fontsize=13, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

# -------------------- Sample Output --------------------
print(f"\n Sample YOLO label lines (first 5):")
for _, r in df_yolo.head(5).iterrows():
    print(f"   {r['file_name']}: {r['yolo_line']}")

<a id='splitting'></a>
## 5. Document-Grouped Dataset Splitting

**Critical:** Split by **document ID** (e.g., `PMC1064076`), not by individual image. Pages from the same paper share identical table styles — placing them across splits causes data leakage and inflates validation scores.

Strategy: `GroupShuffleSplit` with document ID as the group key.
- **Pass 1:** 85% train vs 15% temp
- **Pass 2:** Split temp into ~66.7% val / ~33.3% test → ~10% val, ~5% test overall

In [ ]:
# ============================================================
# Step 5: Document-Grouped Dataset Splitting
# ============================================================

print("="*60)
print(" DOCUMENT-GROUPED DATASET SPLITTING")
print("="*60)

# Build image-level DataFrame with document groups
# (includes ALL images, even those without table annotations)
df_split = df_images[['image_id', 'file_name', 'doc_id']].copy()

# Count table annotations per image
labels_per_image = df_yolo.groupby('image_id').size().reset_index(name='n_labels')
df_split = df_split.merge(labels_per_image, on='image_id', how='left')
df_split['n_labels'] = df_split['n_labels'].fillna(0).astype(int)

n_docs = df_split['doc_id'].nunique()
print(f"\n   Total images: {len(df_split):,}")
print(f"   Unique documents: {n_docs}")
print(f"   Images with tables: {(df_split['n_labels'] > 0).sum():,}")
print(f"   Images without tables: {(df_split['n_labels'] == 0).sum():,}")

# ============================================================
# Pass 1: Train vs Temp (val + test)
# ============================================================
temp_ratio = VAL_RATIO + TEST_RATIO  # 0.15

gss1 = GroupShuffleSplit(n_splits=1, test_size=temp_ratio, random_state=RANDOM_SEED)
train_idx, temp_idx = next(gss1.split(df_split, groups=df_split['doc_id']))

df_train = df_split.iloc[train_idx].copy()
df_temp  = df_split.iloc[temp_idx].copy()

# ============================================================
# Pass 2: Val vs Test from temp
# ============================================================
test_from_temp = TEST_RATIO / temp_ratio  # 0.05/0.15 = 0.3333

gss2 = GroupShuffleSplit(n_splits=1, test_size=test_from_temp, random_state=RANDOM_SEED)
val_idx, test_idx = next(gss2.split(df_temp, groups=df_temp['doc_id']))

df_val  = df_temp.iloc[val_idx].copy()
df_test = df_temp.iloc[test_idx].copy()

# Assign split labels
df_train['split'] = 'train'
df_val['split']   = 'val'
df_test['split']  = 'test'

df_all_splits = pd.concat([df_train, df_val, df_test], ignore_index=True)

# ============================================================
# Validation: Zero Document Overlap
# ============================================================
train_docs = set(df_train['doc_id'].unique())
val_docs   = set(df_val['doc_id'].unique())
test_docs  = set(df_test['doc_id'].unique())

leak_tv = train_docs & val_docs
leak_tt = train_docs & test_docs
leak_vt = val_docs & test_docs

print(f"\n" + "-"*60)
print(f" SPLIT STATISTICS")
print(f"-"*60)

for name, df_s in [('Train', df_train), ('Val', df_val), ('Test', df_test)]:
    n_imgs = len(df_s)
    n_lbls = df_s['n_labels'].sum()
    n_doc  = df_s['doc_id'].nunique()
    pct    = n_imgs / len(df_split) * 100
    density = n_lbls / n_imgs if n_imgs > 0 else 0
    print(f"\n   {name}:")
    print(f"      Images:      {n_imgs:,} ({pct:.1f}%)")
    print(f"      Labels:      {n_lbls:,}")
    print(f"      Documents:   {n_doc}")
    print(f"      Labels/Img:  {density:.2f}")

print(f"\n" + "-"*60)
print(f" DATA LEAKAGE CHECK")
print(f"-"*60)
print(f"   Train-Val overlap:  {len(leak_tv)} docs  {'OK' if len(leak_tv)==0 else 'LEAK DETECTED!'}")
print(f"   Train-Test overlap: {len(leak_tt)} docs  {'OK' if len(leak_tt)==0 else 'LEAK DETECTED!'}")
print(f"   Val-Test overlap:   {len(leak_vt)} docs  {'OK' if len(leak_vt)==0 else 'LEAK DETECTED!'}")

assert len(leak_tv) == 0, "Data leakage between Train and Val!"
assert len(leak_tt) == 0, "Data leakage between Train and Test!"
assert len(leak_vt) == 0, "Data leakage between Val and Test!"

# -------------------- Visualization --------------------
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Split distribution (pie)
ax1 = axes[0]
sizes = [len(df_train), len(df_val), len(df_test)]
labels = [f'Train\n{sizes[0]:,}', f'Val\n{sizes[1]:,}', f'Test\n{sizes[2]:,}']
colors = ['#3498DB', '#2ECC71', '#F39C12']
ax1.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90,
        wedgeprops=dict(edgecolor='black', linewidth=0.5))
ax1.set_title('Image Split Distribution', fontsize=12, fontweight='bold')

# Labels per split (bar)
ax2 = axes[1]
split_names = ['Train', 'Val', 'Test']
split_labels = [df_train['n_labels'].sum(), df_val['n_labels'].sum(), df_test['n_labels'].sum()]
bars = ax2.bar(split_names, split_labels, color=colors, edgecolor='black', linewidth=0.5)
ax2.set_ylabel('Number of Labels', fontsize=11)
ax2.set_title('Table Labels per Split', fontsize=12, fontweight='bold')
for bar, val in zip(bars, split_labels):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
             f'{val:,}', ha='center', va='bottom', fontsize=10)

# Docs per split (bar)
ax3 = axes[2]
split_docs = [len(train_docs), len(val_docs), len(test_docs)]
bars = ax3.bar(split_names, split_docs, color=colors, edgecolor='black', linewidth=0.5)
ax3.set_ylabel('Number of Documents', fontsize=11)
ax3.set_title('Unique Documents per Split', fontsize=12, fontweight='bold')
for bar, val in zip(bars, split_docs):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
             f'{val}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

<a id='export'></a>
## 6. YOLO Directory Export

Creates the standard YOLO directory structure:
```
yolo_dataset/
├── data.yaml
├── images/
│   ├── train/
│   ├── val/
│   └── test/
└── labels/
    ├── train/
    ├── val/
    └── test/
```

In [ ]:
# ============================================================
# Step 6: YOLO Directory Creation & Export
# ============================================================

print("="*60)
print(" YOLO DIRECTORY EXPORT")
print("="*60)

start_time = time.time()

# -------------------- Create Directory Structure --------------------
# Clean previous output if exists
if os.path.exists(OUTPUT_ROOT):
    shutil.rmtree(OUTPUT_ROOT)
    print(f"\n   Removed existing output directory.")

for split in ['train', 'val', 'test']:
    os.makedirs(os.path.join(OUTPUT_ROOT, 'images', split), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_ROOT, 'labels', split), exist_ok=True)

print(f"   Created directory structure at: {OUTPUT_ROOT}")

# -------------------- Build YOLO labels lookup --------------------
# {file_name: [yolo_line1, yolo_line2, ...]}
labels_by_file = defaultdict(list)
for _, r in df_yolo.iterrows():
    labels_by_file[r['file_name']].append(r['yolo_line'])

# -------------------- Export Images & Labels --------------------
export_stats = {'train': {'images': 0, 'labels': 0, 'empty': 0},
                'val':   {'images': 0, 'labels': 0, 'empty': 0},
                'test':  {'images': 0, 'labels': 0, 'empty': 0}}

errors = []

for _, row in df_all_splits.iterrows():
    split     = row['split']
    file_name = row['file_name']
    
    # --- Copy image ---
    src_path = os.path.join(IMAGES_DIR, file_name)
    dst_path = os.path.join(OUTPUT_ROOT, 'images', split, file_name)
    
    if os.path.exists(src_path):
        shutil.copy2(src_path, dst_path)
        export_stats[split]['images'] += 1
    else:
        errors.append(f"Image not found: {src_path}")
        continue
    
    # --- Write label file ---
    label_name = os.path.splitext(file_name)[0] + '.txt'
    label_path = os.path.join(OUTPUT_ROOT, 'labels', split, label_name)
    
    lines = labels_by_file.get(file_name, [])
    
    with open(label_path, 'w') as f:
        if lines:
            f.write('\n'.join(lines) + '\n')
            export_stats[split]['labels'] += len(lines)
        # else: empty file (image with no tables)
    
    if not lines:
        export_stats[split]['empty'] += 1

# -------------------- Generate data.yaml --------------------
yaml_content = textwrap.dedent(f"""\
    # YOLO Dataset Configuration - Table Detection
    # Auto-generated by Phase 2 Pipeline
    
    path: {OUTPUT_ROOT}
    train: images/train
    val: images/val
    test: images/test
    
    # Classes
    nc: {len(CLASS_NAMES)}
    names: {CLASS_NAMES}
""")

yaml_path = os.path.join(OUTPUT_ROOT, 'data.yaml')
with open(yaml_path, 'w') as f:
    f.write(yaml_content)

elapsed = time.time() - start_time

# -------------------- Report --------------------
print(f"\n" + "-"*60)
print(f" EXPORT SUMMARY")
print(f"-"*60)

total_imgs = 0
total_lbls = 0
total_empty = 0

for split in ['train', 'val', 'test']:
    s = export_stats[split]
    print(f"\n   {split.upper()}:")
    print(f"      Images copied:     {s['images']:,}")
    print(f"      Label lines:       {s['labels']:,}")
    print(f"      Empty label files:  {s['empty']}")
    total_imgs += s['images']
    total_lbls += s['labels']
    total_empty += s['empty']

print(f"\n   TOTAL:")
print(f"      Images: {total_imgs:,}")
print(f"      Labels: {total_lbls:,}")
print(f"      Empty label files: {total_empty}")

if errors:
    print(f"\n   Errors ({len(errors)}):")
    for e in errors[:10]:
        print(f"      {e}")
else:
    print(f"\n   No errors during export.")

print(f"\n   data.yaml written to: {yaml_path}")
print(f"   Export completed in {elapsed:.1f}s")

# -------------------- File Count Integrity Check --------------------
print(f"\n" + "-"*60)
print(f" FILE COUNT INTEGRITY CHECK")
print(f"-"*60)

for split in ['train', 'val', 'test']:
    img_dir = os.path.join(OUTPUT_ROOT, 'images', split)
    lbl_dir = os.path.join(OUTPUT_ROOT, 'labels', split)
    n_img_files = len([f for f in os.listdir(img_dir) if f.endswith('.jpg')])
    n_lbl_files = len([f for f in os.listdir(lbl_dir) if f.endswith('.txt')])
    match = "OK" if n_img_files == n_lbl_files else "MISMATCH!"
    print(f"   {split:>5s}: {n_img_files} images, {n_lbl_files} labels  {match}")

# Print data.yaml content
print(f"\n" + "-"*60)
print(f" data.yaml")
print(f"-"*60)
print(yaml_content)

<a id='verification'></a>
## 7. Visual Verification

Load sample images from each split, parse their `.txt` label files, convert YOLO normalized coordinates back to pixel coordinates, and overlay bounding boxes. This is the **end-to-end correctness check**.

In [ ]:
# ============================================================
# Step 7: Visual Verification  (YOLO -> pixel overlay)
# ============================================================

print("="*60)
print(" VISUAL VERIFICATION")
print("="*60)

def yolo_to_pixel(x_center, y_center, w_norm, h_norm, img_w, img_h):
    """Convert YOLO normalized coords back to pixel [x, y, w, h]."""
    w_px = w_norm * img_w
    h_px = h_norm * img_h
    x_px = x_center * img_w - w_px / 2
    y_px = y_center * img_h - h_px / 2
    return x_px, y_px, w_px, h_px

def parse_yolo_label(label_path):
    """Parse a YOLO .txt label file and return list of (class_id, xc, yc, w, h)."""
    boxes = []
    if not os.path.exists(label_path):
        return boxes
    with open(label_path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            cls = int(parts[0])
            xc, yc, w, h = map(float, parts[1:5])
            boxes.append((cls, xc, yc, w, h))
    return boxes

# Select 2 random images from each split (that have labels)
np.random.seed(RANDOM_SEED)
sample_images = []

for split_name, df_s in [('train', df_train), ('val', df_val), ('test', df_test)]:
    # Prefer images with tables for more informative visualization
    with_labels = df_s[df_s['n_labels'] > 0]
    if len(with_labels) >= 2:
        chosen = with_labels.sample(2, random_state=RANDOM_SEED)
    else:
        chosen = df_s.sample(min(2, len(df_s)), random_state=RANDOM_SEED)
    
    for _, row in chosen.iterrows():
        sample_images.append((split_name, row['file_name']))

n_samples = len(sample_images)
n_cols = min(3, n_samples)
n_rows = (n_samples + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 6 * n_rows))
if n_samples == 1:
    axes = np.array([[axes]])
elif n_rows == 1:
    axes = axes.reshape(1, -1)

for idx, (split_name, file_name) in enumerate(sample_images):
    r, c = idx // n_cols, idx % n_cols
    ax = axes[r, c]
    
    img_path = os.path.join(OUTPUT_ROOT, 'images', split_name, file_name)
    label_name = os.path.splitext(file_name)[0] + '.txt'
    label_path = os.path.join(OUTPUT_ROOT, 'labels', split_name, label_name)
    
    if os.path.exists(img_path):
        img = Image.open(img_path)
        img_w, img_h = img.size
        ax.imshow(img, cmap='gray')
        
        # Parse and overlay YOLO boxes
        boxes = parse_yolo_label(label_path)
        for cls_id, xc, yc, w, h in boxes:
            px, py, pw, ph = yolo_to_pixel(xc, yc, w, h, img_w, img_h)
            rect = Rectangle((px, py), pw, ph, linewidth=2.5,
                             edgecolor=TABLE_COLOR, facecolor='none',
                             linestyle='-')
            ax.add_patch(rect)
            ax.text(px, py - 8, f'table ({w*img_w:.0f}x{h*img_h:.0f}px)',
                    color=TABLE_COLOR, fontsize=9, fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8))
        
        ax.set_title(f'[{split_name.upper()}] {file_name}\n{len(boxes)} table(s)', fontsize=10)
    else:
        ax.text(0.5, 0.5, f'Image not found:\n{file_name}',
                ha='center', va='center', transform=ax.transAxes)
    ax.axis('off')

# Hide unused subplots
for idx in range(n_samples, n_rows * n_cols):
    r, c = idx // n_cols, idx % n_cols
    axes[r, c].axis('off')

fig.suptitle('Visual Verification: YOLO Labels Overlaid on Images',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\n   Visual verification complete. Inspect the overlaid boxes above.")
print("   Red rectangles should tightly enclose each table in the image.")

<a id='summary'></a>
## 8. Summary Report & Completion Checklist

In [ ]:
# ============================================================
# Step 8: Summary Report & Completion Checklist
# ============================================================

print("="*70)
print(" PHASE 2 SUMMARY REPORT")
print("="*70)

# -------------------- Pipeline Summary --------------------
print("\n" + "-"*70)
print(" PIPELINE STATISTICS")
print("-"*70)

summary_rows = [
    ['Total COCO annotations (input)',        f'{n_total_anns:,}'],
    ['Non-table annotations dropped',         f'{n_dropped:,}'],
    ['Table annotations (cat 1) retained',    f'{n_table_anns_before:,}'],
    ['Table bboxes expanded (safety net)',     f'{expansion_count}'],
    ['Boxes clipped to [0,1]',                f'{n_clipped}'],
    ['Boxes discarded (degenerate)',           f'{n_discarded}'],
    ['Valid YOLO labels (output)',             f'{n_after:,}'],
    ['', ''],
    ['Total images',                          f'{len(df_split):,}'],
    ['Unique documents',                      f'{n_docs}'],
    ['Train images',                          f'{len(df_train):,} ({len(df_train)/len(df_split)*100:.1f}%)'],
    ['Val images',                            f'{len(df_val):,} ({len(df_val)/len(df_split)*100:.1f}%)'],
    ['Test images',                           f'{len(df_test):,} ({len(df_test)/len(df_split)*100:.1f}%)'],
    ['Train documents',                       f'{len(train_docs)}'],
    ['Val documents',                         f'{len(val_docs)}'],
    ['Test documents',                        f'{len(test_docs)}'],
    ['Document overlap (leakage)',            f'None'],
]

df_summary = pd.DataFrame(summary_rows, columns=['Metric', 'Value'])
display(df_summary.style.hide(axis='index'))

# -------------------- Annotation Reduction Funnel --------------------
print("\n" + "-"*70)
print(" ANNOTATION REDUCTION FUNNEL")
print("-"*70)

funnel_stages = [
    ('Raw COCO annotations',    n_total_anns),
    ('After semantic filtering', n_table_anns_before),
    ('After bbox expansion',     n_table_anns_before),  # same count, expanded coords
    ('After sanitization',       n_after),
]

fig, ax = plt.subplots(figsize=(10, 4))
stage_names = [s[0] for s in funnel_stages]
stage_counts = [s[1] for s in funnel_stages]
colors_funnel = ['#E74C3C', '#F39C12', '#3498DB', '#2ECC71']

bars = ax.barh(stage_names[::-1], stage_counts[::-1], color=colors_funnel[::-1],
               edgecolor='black', linewidth=0.5, height=0.6)
ax.set_xlabel('Number of Annotations', fontsize=12)
ax.set_title('Annotation Reduction Funnel', fontsize=13, fontweight='bold')

for bar, val in zip(bars, stage_counts[::-1]):
    ax.text(bar.get_width() + 200, bar.get_y() + bar.get_height()/2,
            f'{val:,}', ha='left', va='center', fontsize=11, fontweight='bold')

ax.set_xlim(0, max(stage_counts) * 1.15)
plt.tight_layout()
plt.show()

# -------------------- YOLO Label Statistics --------------------
print("\n" + "-"*70)
print(" YOLO LABEL COORDINATE STATISTICS")
print("-"*70)

yolo_stats = df_yolo[['x_center', 'y_center', 'w_norm', 'h_norm']].describe()
display(yolo_stats.round(6))

# -------------------- Final Integrity Checks --------------------
print("\n" + "="*70)
print(" PHASE 2 COMPLETION CHECKLIST")
print("="*70)

# Compute checks
all_coords_valid = (df_yolo[['x_center','y_center','w_norm','h_norm']].min().min() >= 0 and
                    df_yolo[['x_center','y_center','w_norm','h_norm']].max().max() <= 1)

checklist = [
    ("Semantic filtering (table-only)",          True),
    ("Table bbox expansion safety net",          True),
    ("COCO -> YOLO coordinate conversion",       True),
    ("Coordinate clipping [0,1]",                all_coords_valid),
    ("Degenerate box removal",                   True),
    ("Document-grouped splitting (no leakage)",  len(leak_tv) == 0 and len(leak_tt) == 0 and len(leak_vt) == 0),
    ("Train/Val/Test directories created",       all(os.path.isdir(os.path.join(OUTPUT_ROOT, 'images', s)) for s in ['train','val','test'])),
    ("Label files exported",                     total_lbls > 0),
    ("Image-label count match",                  all(
        len(os.listdir(os.path.join(OUTPUT_ROOT, 'images', s))) == 
        len(os.listdir(os.path.join(OUTPUT_ROOT, 'labels', s)))
        for s in ['train','val','test']
    )),
    ("data.yaml generated",                      os.path.exists(yaml_path)),
    ("Visual verification complete",             True),
]

all_passed = True
for task, passed in checklist:
    status = "OK" if passed else "FAIL"
    if not passed:
        all_passed = False
    print(f"   {status}  {task}")

print("\n" + "="*70)
if all_passed:
    print(" ALL CHECKS PASSED - READY FOR PHASE 3: MODEL TRAINING")
else:
    print(" SOME CHECKS FAILED - REVIEW ABOVE BEFORE PROCEEDING")
print("="*70)

print(f"\n   Output directory: {OUTPUT_ROOT}")
print(f"   data.yaml: {yaml_path}")
print(f"\n   Next steps:")
print(f"   1. Use data.yaml as input to YOLOv10 training")
print(f"   2. Consider adding hard-negative empty images in a future phase")
print(f"   3. Proceed to Phase 3: Model Training & Evaluation")

# Phase 3 & 4: YOLOv10 Domain-Specific Training & Smart Inference

**Objective:** Train YOLOv10-Medium for table detection on document images, then run inference with intelligent post-processing (box dilation, heuristic filtering).

---

## Table of Contents

### Phase 3: Domain-Specific Training Pipeline
1. [Environment Setup & Installation](#setup)
2. [Configuration](#config)
3. [Custom Augmentation Pipeline](#augmentation)
4. [Model Initialization & Training](#training)
5. [Per-Epoch Metrics](#epoch-metrics)
6. [Training Visualization](#train-viz)
7. [Test Set Evaluation](#test-eval)

### Phase 4: Inference & Smart Post-Processing
8. [Load Best Weights & Run Inference](#inference)
9. [Box Dilation (Padding)](#dilation)
10. [Heuristic Filtering](#filtering)
11. [End-to-End Inference Pipeline](#pipeline)
12. [Post-Processing Evaluation](#post-eval)
13. [Results Visualization](#results-viz)
14. [Export Results](#export)

---

### Key Enhancements Over Standard YOLO Training

| Enhancement | Standard | This Pipeline |
|---|---|---|
| Resolution | 640×640 squish | 1024×1024 letterbox |
| Mosaic | ON | **OFF** (cross-pattern mimics tables) |
| MixUp | ON | **OFF** (ghosting hurts borders) |
| HSV/Color Jitter | ON | **OFF** (structure > color) |
| Flip LR | ON | **OFF** (reading order) |
| CLAHE | OFF | **ON** (contrast on dark scans) |
| Grayscale Aug | OFF | **ON** (learn structure, not color) |
| Micro-rotation | OFF | **ON** (±0.5° realistic scan skew) |
| Post-processing | Raw boxes | Dilation + AR filter + nested removal |

<a id='setup'></a>
## 1. Environment Setup & Installation

In [ ]:
# ============================================================
# Cell 1: Environment Setup & Installation
# ============================================================
import subprocess
import sys

def pip_install(package, flags=None):
    """Install a pip package with optional flags."""
    cmd = [sys.executable, '-m', 'pip', 'install']
    if flags:
        cmd.extend(flags)
    cmd.append(package)
    subprocess.check_call(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print("="*60)
print(" ENVIRONMENT SETUP")
print("="*60)

# 1. Install YOLOv10 from THU-MIG repo (includes ultralytics fork)
print("\n[1/2] Installing YOLOv10 (THU-MIG)...")
pip_install('git+https://github.com/THU-MIG/yolov10.git', ['-q'])
print("      Done.")

# 2. Install albumentations for custom augmentation pipeline
print("[2/2] Installing albumentations...")
pip_install('albumentations', ['-q', '--upgrade'])
print("      Done.")

# 3. Verify GPU availability
import torch
print(f"\n PyTorch version: {torch.__version__}")
print(f" CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f" GPU:             {gpu_name}")
    print(f" VRAM:            {vram_gb:.1f} GB")
else:
    print("  No GPU detected. Training will be extremely slow on CPU!")

# 4. Verify imports
from ultralytics import YOLOv10
import albumentations as A
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json, os, shutil, warnings, time
from collections import defaultdict
from PIL import Image
from matplotlib.patches import Rectangle

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='husl')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print("\n All imports successful.")
print("="*60)

<a id='config'></a>
## 2. Configuration

**Model:** YOLOv10-Medium — better thin-line feature extraction than Small/Nano, fits P100 16GB VRAM.

**Resolution:** 1024×1024 letterbox — documents are usually A4 (portrait). Squishing to square distorts aspect ratio. Letterboxing adds gray padding bars to make the image square without stretching text/lines. 1024px is needed because table borders are often only 1–2 pixels thick; lower resolution makes them disappear.

In [ ]:
# ============================================================
# Cell 2: Configuration
# ============================================================

# =====================================================================
# PATHS  (Kaggle environment — output of Phase 2)
# =====================================================================
DATA_YAML    = '/kaggle/working/yolo_dataset/data.yaml'
IMAGES_DIR   = '/kaggle/working/yolo_dataset/images'
LABELS_DIR   = '/kaggle/working/yolo_dataset/labels'
OUTPUT_DIR   = '/kaggle/working'
CROPS_DIR    = '/kaggle/working/cropped_tables'

# =====================================================================
# MODEL
# =====================================================================
MODEL_VARIANT  = 'yolov10m.pt'     # YOLOv10-Medium
IMGSZ          = 1024              # Letterbox resolution (preserves A4 aspect ratio)

# =====================================================================
# TRAINING HYPERPARAMETERS
# =====================================================================
EPOCHS         = 60                # Max epochs (early stopping may terminate earlier)
PATIENCE       = 10                # Early stopping patience (epochs without improvement)
BATCH_SIZE     = 8                 # Conservative for P100 16GB at 1024px (~12-14GB VRAM)
OPTIMIZER      = 'AdamW'           # Better generalization than SGD for small datasets
LR0            = 1e-3              # Initial learning rate
LRF            = 0.01              # Final LR factor (cosine decay: LR0 * LRF = 1e-5)
WEIGHT_DECAY   = 5e-4              # AdamW L2 regularization
WARMUP_EPOCHS  = 3                 # Gradual LR ramp-up
LABEL_SMOOTHING = 0.01             # Slight smoothing for noisy annotations
BOX_LOSS_GAIN  = 7.5               # Increase box loss weight (precise localization critical)

# =====================================================================
# AUGMENTATION (Document-Specific Strategy)
# =====================================================================
# DISABLED — harmful for document analysis
AUG_MOSAIC     = 0.0   # OFF: cross-pattern mimics table intersections → false positives
AUG_MIXUP      = 0.0   # OFF: ghosting damages border detection
AUG_HSV_H      = 0.0   # OFF: documents are structure-based, not color-based
AUG_HSV_S      = 0.0   # OFF
AUG_HSV_V      = 0.0   # OFF
AUG_FLIPLR     = 0.0   # OFF: preserve text reading order
AUG_FLIPUD     = 0.0   # OFF: unrealistic for documents

# ENABLED — beneficial for document analysis
AUG_DEGREES    = 0.5   # ±0.5° micro-rotation (realistic scanner skew)
AUG_SCALE      = 0.2   # ±20% mild scale jitter

# CLAHE / Grayscale — injected via Albumentations (see Cell 3)
AUG_CLAHE_P    = 0.3   # 30% chance of CLAHE (contrast enhancement for dark scans)
AUG_GRAY_P     = 0.2   # 20% chance of grayscale (forces structural learning)

# =====================================================================
# INFERENCE & POST-PROCESSING (Phase 4)
# =====================================================================
CONF_THRESH    = 0.25              # Confidence threshold
BOX_PADDING_PX = 15                # Dilation padding in pixels
AR_MIN         = 0.1               # Min aspect ratio (w/h) — discard < 1:10
AR_MAX         = 10.0              # Max aspect ratio (w/h) — discard > 10:1
MIN_AREA_PCT   = 0.01              # Min box area as % of image (discard < 1%)
NESTED_IOU     = 0.85              # IoU threshold for nested box detection
NMS_IOU_THRESH = 0.5               # Light NMS for duplicate safety check

# =====================================================================
# REPRODUCIBILITY
# =====================================================================
RANDOM_SEED    = 42

# =====================================================================
# CLASS INFO
# =====================================================================
CLASS_NAMES    = ['table']
NC             = 1

print("="*60)
print(" CONFIGURATION SUMMARY")
print("="*60)
print(f"\n Model:      {MODEL_VARIANT}")
print(f" Resolution: {IMGSZ}x{IMGSZ} (letterbox)")
print(f" Epochs:     {EPOCHS} (patience={PATIENCE})")
print(f" Batch Size: {BATCH_SIZE}")
print(f" Optimizer:  {OPTIMIZER} (lr={LR0}, wd={WEIGHT_DECAY})")
print(f" Box Loss:   {BOX_LOSS_GAIN}")
print(f"\n Augmentation Strategy:")
print(f"   Mosaic:  {AUG_MOSAIC}  (OFF)")
print(f"   MixUp:   {AUG_MIXUP}  (OFF)")
print(f"   HSV:     {AUG_HSV_H}/{AUG_HSV_S}/{AUG_HSV_V}  (OFF)")
print(f"   Flip:    LR={AUG_FLIPLR}, UD={AUG_FLIPUD}  (OFF)")
print(f"   Rotate:  ±{AUG_DEGREES}°  (ON)")
print(f"   Scale:   ±{AUG_SCALE*100:.0f}%  (ON)")
print(f"   CLAHE:   p={AUG_CLAHE_P}  (ON)")
print(f"   Gray:    p={AUG_GRAY_P}  (ON)")
print(f"\n Post-Processing:")
print(f"   Confidence:  {CONF_THRESH}")
print(f"   Box Padding: {BOX_PADDING_PX}px")
print(f"   AR Range:    [{AR_MIN}, {AR_MAX}]")
print(f"   Min Area:    {MIN_AREA_PCT*100:.0f}% of image")
print(f"\n Data YAML:  {DATA_YAML}")
print("="*60)

<a id='augmentation'></a>
## 3. Custom Augmentation Pipeline (Albumentations)

**Strategy:** Monkey-patch the ultralytics `Albumentations` wrapper class to inject document-specific augmentations:
- **CLAHE** (`p=0.3`): Contrast Limited Adaptive Histogram Equalization — improves contrast on scans with dark/shadowy backgrounds
- **Grayscale** (`p=0.2`): Forces the model to learn structural features (lines, borders) instead of relying on header colors

The built-in harmful augmentations (mosaic, mixup, HSV, flips) are disabled via `model.train()` parameters.

In [ ]:
# ============================================================
# Cell 3: Custom Augmentation Pipeline — Albumentations Injection
# ============================================================

import albumentations as A
from ultralytics.data import augment as ultralytics_augment

print("="*60)
print(" CUSTOM AUGMENTATION PIPELINE")
print("="*60)

def _patched_albumentations_init(self, p=1.0):
    """
    Replace the default Albumentations pipeline with document-specific transforms.
    
    Injected transforms:
    - CLAHE: Contrast enhancement for dark/shadowy scans
    - ToGray: Force structural learning (lines/borders over colors)
    
    All other ultralytics augmentations (mosaic, mixup, HSV, flip)
    are controlled via model.train() parameters — not here.
    """
    self.p = p
    self.transform = None

    try:
        transforms = [
            A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=AUG_CLAHE_P),
            A.ToGray(p=AUG_GRAY_P),
        ]

        self.transform = A.Compose(
            transforms,
            bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels'])
        )

        transform_names = ', '.join(t.__class__.__name__ for t in transforms)
        print(f"   Albumentations pipeline injected: [{transform_names}]")

    except Exception as e:
        print(f"   Albumentations injection failed: {e}")
        self.transform = None

# Apply the monkey-patch
ultralytics_augment.Albumentations.__init__ = _patched_albumentations_init

print("\n Monkey-patch applied to ultralytics.data.augment.Albumentations")
print(" The following augmentations will be applied during training:")
print(f"   - CLAHE (clip_limit=2.0, p={AUG_CLAHE_P})")
print(f"   - ToGray (p={AUG_GRAY_P})")
print("\n The following augmentations are DISABLED via model.train() params:")
print("   - Mosaic (cross-pattern mimics table intersections)")
print("   - MixUp (ghosting damages border detection)")
print("   - HSV/Color Jitter (documents are structure-based)")
print("   - Horizontal/Vertical Flip (text reading order)")
print("="*60)

<a id='training'></a>
## 4. Model Initialization & Training

**Model:** YOLOv10-M (Medium) — fits P100 16GB VRAM, superior feature extraction for thin grid lines compared to Nano/Small variants.

**Training Strategy:**
- **AdamW** optimizer with cosine LR decay ($10^{-3} \to 10^{-5}$)
- **3-epoch warmup** for gradient stabilization
- **Early stopping** (patience=10) to prevent overfitting on ~1500 samples
- **Box loss gain = 7.5** (default is 7.5) — precise table localization is paramount
- **Label smoothing = 0.01** — accounts for annotation noise found in Phase 1

In [ ]:
# ============================================================
# Cell 4: Model Initialization & Training
# ============================================================

import urllib.request
import os
import time

# -------------------- CRITICAL: Disable WandB BEFORE importing ultralytics --------------------
# WandB reads this at import time, so it must be set first
os.environ['WANDB_MODE'] = 'disabled'
os.environ['WANDB_DISABLED'] = 'true'
os.environ['WANDB_SILENT'] = 'true'

import torch
from ultralytics import YOLOv10
from ultralytics.nn import tasks as ultralytics_tasks

print("="*60)
print(" MODEL INITIALIZATION & TRAINING")
print("="*60)

# -------------------- Download Weights --------------------
WEIGHTS_URL = 'https://github.com/THU-MIG/yolov10/releases/download/v1.1/yolov10m.pt'
if not os.path.exists(MODEL_VARIANT):
    print(f"\n Downloading {MODEL_VARIANT} from {WEIGHTS_URL}...")
    try:
        urllib.request.urlretrieve(WEIGHTS_URL, MODEL_VARIANT)
        print("   Download complete.")
    except Exception as e:
        print(f"   Download failed: {e}")
        print("   Please manually download the weights or check internet connection.")
else:
    print(f"\n Found local weights: {MODEL_VARIANT}")

# -------------------- Fix PyTorch 2.6+ Safe Load Issue --------------------
# Patch the specific ultralytics function that calls torch.load
print("\n Patching ultralytics torch_safe_load for PyTorch 2.6+ compatibility...")

# Save the original function if not already saved
if not hasattr(ultralytics_tasks, '_original_torch_safe_load'):
    ultralytics_tasks._original_torch_safe_load = ultralytics_tasks.torch_safe_load

# Create a wrapper that forces weights_only=False
def _patched_torch_safe_load(file, device=None):
    """Patched version of torch_safe_load that uses weights_only=False."""
    import torch
    from pathlib import Path
    
    # Simplified logic: just load with weights_only=False
    try:
        ckpt = torch.load(file, map_location=device or "cpu", weights_only=False)
        return ckpt, file
    except Exception as e:
        # Fallback to original function if something goes wrong
        print(f"   Warning: Patched load failed, trying original: {e}")
        return ultralytics_tasks._original_torch_safe_load(file, device)

# Apply the patch
ultralytics_tasks.torch_safe_load = _patched_torch_safe_load
print("   Patch applied to ultralytics.nn.tasks.torch_safe_load")

# -------------------- Initialize Model --------------------
print(f"\n Loading pretrained {MODEL_VARIANT}...")
if os.path.exists(MODEL_VARIANT):
    model = YOLOv10(MODEL_VARIANT)
    print(f"   Model loaded: YOLOv10-Medium")
    print(f"   Pretrained on COCO — fine-tuning for table detection")
else:
    raise FileNotFoundError(f"Weights file {MODEL_VARIANT} not found. Download failed.")

# -------------------- Verify data.yaml --------------------
assert os.path.exists(DATA_YAML), f"data.yaml not found at {DATA_YAML}"
print(f"   data.yaml verified: {DATA_YAML}")

# -------------------- Train --------------------
print(f"\n Starting training...")
print(f"   Epochs:     {EPOCHS} (early stopping patience={PATIENCE})")
print(f"   Resolution: {IMGSZ}x{IMGSZ}")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   Optimizer:  {OPTIMIZER}")
print("-"*60)

train_start = time.time()

results = model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH_SIZE,
    
    # Optimizer
    optimizer=OPTIMIZER,
    lr0=LR0,
    lrf=LRF,
    weight_decay=WEIGHT_DECAY,
    warmup_epochs=WARMUP_EPOCHS,
    
    # Loss weights
    box=BOX_LOSS_GAIN,
    label_smoothing=LABEL_SMOOTHING,
    
    # Augmentation — DISABLED (document-specific)
    mosaic=AUG_MOSAIC,
    mixup=AUG_MIXUP,
    hsv_h=AUG_HSV_H,
    hsv_s=AUG_HSV_S,
    hsv_v=AUG_HSV_V,
    fliplr=AUG_FLIPLR,
    flipud=AUG_FLIPUD,
    
    # Augmentation — ENABLED
    degrees=AUG_DEGREES,
    scale=AUG_SCALE,
    
    # Mosaic close epoch (already disabled, but set to 0 for safety)
    close_mosaic=0,
    
    # Early stopping
    patience=PATIENCE,
    
    # Reproducibility
    seed=RANDOM_SEED,
    deterministic=True,
    
    # Output
    project='runs',
    name='table_detection',
    exist_ok=True,
    verbose=True,
    plots=True,
    save=True,
    val=True,
)

train_elapsed = time.time() - train_start

# -------------------- Identify output paths --------------------
TRAIN_DIR = os.path.join(OUTPUT_DIR, 'runs', 'detect', 'table_detection')
BEST_WEIGHTS = os.path.join(TRAIN_DIR, 'weights', 'best.pt')
LAST_WEIGHTS = os.path.join(TRAIN_DIR, 'weights', 'last.pt')
RESULTS_CSV = os.path.join(TRAIN_DIR, 'results.csv')

print(f"\n Training complete in {train_elapsed/60:.1f} minutes.")
print(f"   Best weights: {BEST_WEIGHTS}")
print(f"   Last weights: {LAST_WEIGHTS}")
print(f"   Results CSV:  {RESULTS_CSV}")

<a id='epoch-metrics'></a>
## 5. Per-Epoch Metrics

Parse `results.csv` from the training run and display a formatted table with all metrics per epoch. Compute and append the F1-Score:

$$F_1 = \frac{2 \cdot \text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$

In [ ]:
# ============================================================
# Cell 5: Per-Epoch Metrics Display
# ============================================================

print("="*60)
print(" PER-EPOCH TRAINING METRICS")
print("="*60)

# -------------------- Load results.csv --------------------
assert os.path.exists(RESULTS_CSV), f"Results CSV not found: {RESULTS_CSV}"

df_results = pd.read_csv(RESULTS_CSV)
df_results.columns = df_results.columns.str.strip()  # Clean whitespace from column names

print(f"\n   Loaded {len(df_results)} epochs from results.csv")
print(f"   Columns: {list(df_results.columns)}")

# -------------------- Rename columns for readability --------------------
# Ultralytics results.csv has columns like:
#   epoch, train/box_loss, train/cls_loss, train/dfl_loss,
#   metrics/precision(B), metrics/recall(B), metrics/mAP50(B), metrics/mAP50-95(B),
#   val/box_loss, val/cls_loss, val/dfl_loss, lr/pg0, lr/pg1, lr/pg2

col_map = {}
for col in df_results.columns:
    c = col.strip()
    if c == 'epoch':
        col_map[col] = 'Epoch'
    elif 'train/box_loss' in c:
        col_map[col] = 'Train Box Loss'
    elif 'train/cls_loss' in c:
        col_map[col] = 'Train Cls Loss'
    elif 'train/dfl_loss' in c:
        col_map[col] = 'Train DFL Loss'
    elif 'val/box_loss' in c:
        col_map[col] = 'Val Box Loss'
    elif 'val/cls_loss' in c:
        col_map[col] = 'Val Cls Loss'
    elif 'val/dfl_loss' in c:
        col_map[col] = 'Val DFL Loss'
    elif 'precision' in c.lower():
        col_map[col] = 'Precision'
    elif 'recall' in c.lower():
        col_map[col] = 'Recall'
    elif 'mAP50-95' in c or 'mAP50_95' in c:
        col_map[col] = 'mAP@0.5:0.95'
    elif 'mAP50' in c:
        col_map[col] = 'mAP@0.5'
    elif 'lr/pg0' in c:
        col_map[col] = 'LR'

df_metrics = df_results.rename(columns=col_map)

# -------------------- Compute F1-Score --------------------
if 'Precision' in df_metrics.columns and 'Recall' in df_metrics.columns:
    P = df_metrics['Precision']
    R = df_metrics['Recall']
    df_metrics['F1-Score'] = np.where(
        (P + R) > 0,
        2 * P * R / (P + R),
        0.0
    )

# -------------------- Epoch column as int --------------------
if 'Epoch' in df_metrics.columns:
    df_metrics['Epoch'] = df_metrics['Epoch'].astype(int)

# -------------------- Select display columns --------------------
display_cols = [c for c in [
    'Epoch', 'Train Box Loss', 'Train Cls Loss', 'Train DFL Loss',
    'Val Box Loss', 'Val Cls Loss', 'Val DFL Loss',
    'Precision', 'Recall', 'mAP@0.5', 'mAP@0.5:0.95', 'F1-Score', 'LR'
] if c in df_metrics.columns]

df_display = df_metrics[display_cols].copy()

# -------------------- Display full table --------------------
print("\n" + "-"*60)
print(" FULL METRICS TABLE")
print("-"*60)

# Format numeric columns
fmt_cols = [c for c in display_cols if c != 'Epoch']
styled = df_display.style.format(
    {c: '{:.4f}' for c in fmt_cols}
).hide(axis='index')

display(styled)

# -------------------- Best Epoch --------------------
if 'mAP@0.5:0.95' in df_metrics.columns:
    best_idx = df_metrics['mAP@0.5:0.95'].idxmax()
    best_epoch = df_metrics.loc[best_idx]
    
    print("\n" + "-"*60)
    print(" BEST EPOCH (by mAP@0.5:0.95)")
    print("-"*60)
    print(f"   Epoch:         {int(best_epoch.get('Epoch', best_idx))}")
    print(f"   Precision:     {best_epoch.get('Precision', 0):.4f}")
    print(f"   Recall:        {best_epoch.get('Recall', 0):.4f}")
    print(f"   mAP@0.5:       {best_epoch.get('mAP@0.5', 0):.4f}")
    print(f"   mAP@0.5:0.95:  {best_epoch.get('mAP@0.5:0.95', 0):.4f}")
    print(f"   F1-Score:      {best_epoch.get('F1-Score', 0):.4f}")

# -------------------- Early Stopping Info --------------------
actual_epochs = len(df_metrics)
print(f"\n   Training ran for {actual_epochs} epochs (max={EPOCHS}, patience={PATIENCE})")
if actual_epochs < EPOCHS:
    print(f"    Early stopping triggered at epoch {actual_epochs}")
else:
    print(f"   Completed all {EPOCHS} epochs (no early stop)")

print("="*60)

<a id='train-viz'></a>
## 6. Training Visualization

Four-panel visualization of training dynamics:
1. **Loss Curves** — Train vs Val for Box/Cls/DFL losses (convergence check)
2. **Detection Metrics** — Precision, Recall, mAP@0.5, mAP@0.5:0.95 over epochs
3. **F1-Score Curve** — Harmonic mean of Precision and Recall
4. **Learning Rate Schedule** — Warmup + cosine decay verification

In [ ]:
# ============================================================
# Cell 6: Training Visualization
# ============================================================

print("="*60)
print(" TRAINING VISUALIZATION")
print("="*60)

epochs = df_metrics['Epoch'] if 'Epoch' in df_metrics.columns else range(len(df_metrics))

# =====================================================================
# PLOT 1: Loss Curves (2x2 grid — Train vs Val for Box/Cls/DFL + Combined)
# =====================================================================
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Box Loss
ax = axes[0, 0]
if 'Train Box Loss' in df_metrics.columns:
    ax.plot(epochs, df_metrics['Train Box Loss'], 'b-', linewidth=2, label='Train', alpha=0.8)
if 'Val Box Loss' in df_metrics.columns:
    ax.plot(epochs, df_metrics['Val Box Loss'], 'r--', linewidth=2, label='Val', alpha=0.8)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Box Loss', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Cls Loss
ax = axes[0, 1]
if 'Train Cls Loss' in df_metrics.columns:
    ax.plot(epochs, df_metrics['Train Cls Loss'], 'b-', linewidth=2, label='Train', alpha=0.8)
if 'Val Cls Loss' in df_metrics.columns:
    ax.plot(epochs, df_metrics['Val Cls Loss'], 'r--', linewidth=2, label='Val', alpha=0.8)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Classification Loss', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# DFL Loss
ax = axes[1, 0]
if 'Train DFL Loss' in df_metrics.columns:
    ax.plot(epochs, df_metrics['Train DFL Loss'], 'b-', linewidth=2, label='Train', alpha=0.8)
if 'Val DFL Loss' in df_metrics.columns:
    ax.plot(epochs, df_metrics['Val DFL Loss'], 'r--', linewidth=2, label='Val', alpha=0.8)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('DFL Loss (Distribution Focal Loss)', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

# Combined Total Loss
ax = axes[1, 1]
train_losses = []
val_losses = []
for prefix, target in [('Train', train_losses), ('Val', val_losses)]:
    total = None
    for loss_type in ['Box Loss', 'Cls Loss', 'DFL Loss']:
        col = f'{prefix} {loss_type}'
        if col in df_metrics.columns:
            total = df_metrics[col] if total is None else total + df_metrics[col]
    if total is not None:
        target.extend(total.tolist())
        style = 'b-' if prefix == 'Train' else 'r--'
        ax.plot(epochs, total, style, linewidth=2, label=f'{prefix} Total', alpha=0.8)

ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Total Loss (Box + Cls + DFL)', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)

fig.suptitle('Training Loss Curves', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# =====================================================================
# PLOT 2: Detection Metrics Over Epochs
# =====================================================================
fig, ax = plt.subplots(figsize=(12, 5))

metric_configs = [
    ('Precision', '#E74C3C', '-'),
    ('Recall', '#3498DB', '-'),
    ('mAP@0.5', '#2ECC71', '-'),
    ('mAP@0.5:0.95', '#9B59B6', '--'),
]

for col_name, color, style in metric_configs:
    if col_name in df_metrics.columns:
        ax.plot(epochs, df_metrics[col_name], color=color, linestyle=style,
                linewidth=2, label=col_name, alpha=0.85)

ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Detection Metrics Over Epochs', fontsize=13, fontweight='bold')
ax.legend(fontsize=11, loc='lower right')
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# =====================================================================
# PLOT 3: F1-Score Curve
# =====================================================================
if 'F1-Score' in df_metrics.columns:
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(epochs, df_metrics['F1-Score'], color='#F39C12', linewidth=2.5, 
            label='F1-Score', alpha=0.9)
    ax.fill_between(epochs, df_metrics['F1-Score'], alpha=0.15, color='#F39C12')
    
    # Mark best F1
    best_f1_idx = df_metrics['F1-Score'].idxmax()
    best_f1_epoch = epochs.iloc[best_f1_idx] if hasattr(epochs, 'iloc') else best_f1_idx
    best_f1_val = df_metrics['F1-Score'].iloc[best_f1_idx]
    ax.axvline(x=best_f1_epoch, color='red', linestyle=':', alpha=0.5)
    ax.scatter([best_f1_epoch], [best_f1_val], color='red', s=100, zorder=5, 
               label=f'Best F1={best_f1_val:.4f} (Epoch {int(best_f1_epoch)})')
    
    ax.set_xlabel('Epoch', fontsize=12)
    ax.set_ylabel('F1-Score', fontsize=12)
    ax.set_title('F1-Score Over Epochs', fontsize=13, fontweight='bold')
    ax.legend(fontsize=11)
    ax.set_ylim(0, 1.05)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# =====================================================================
# PLOT 4: Learning Rate Schedule
# =====================================================================
if 'LR' in df_metrics.columns:
    fig, ax = plt.subplots(figsize=(12, 3.5))
    ax.plot(epochs, df_metrics['LR'], color='#1ABC9C', linewidth=2, label='Learning Rate (pg0)')
    ax.set_xlabel('Epoch', fontsize=12)
    ax.set_ylabel('Learning Rate', fontsize=12)
    ax.set_title('Learning Rate Schedule (Warmup + Cosine Decay)', fontsize=13, fontweight='bold')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    ax.ticklabel_format(axis='y', style='scientific', scilimits=(-4, -4))
    plt.tight_layout()
    plt.show()

print("   All training plots generated.")
print("="*60)

<a id='test-eval'></a>
## 7. Test Set Evaluation

Evaluate the best model checkpoint on the held-out **test split** (5% of data, document-grouped — zero leakage from train/val).

**Metrics:**
- **IoU** (Intersection over Union) — localization quality
- **mAP@0.5** and **mAP@0.5:0.95** — detection accuracy
- **Precision** and **Recall** — completeness vs. correctness
- **F1-Score** — harmonic mean of P and R

In [ ]:
# ============================================================
# Cell 7: Test Set Evaluation
# ============================================================

print("="*60)
print(" TEST SET EVALUATION (best.pt)")
print("="*60)

# -------------------- Load Best Weights --------------------
print(f"\n Loading best weights: {BEST_WEIGHTS}")
model_eval = YOLOv10(BEST_WEIGHTS)

# -------------------- Run Validation on Test Split --------------------
print(f" Evaluating on test split...")

test_results = model_eval.val(
    data=DATA_YAML,
    split='test',
    imgsz=IMGSZ,
    conf=CONF_THRESH,
    batch=BATCH_SIZE,
    plots=True,
    save_json=True,
    verbose=True,
)

# -------------------- Extract Metrics --------------------
# test_results.box contains: mp, mr, map50, map (map50-95)
test_precision = test_results.box.mp       # Mean Precision
test_recall    = test_results.box.mr       # Mean Recall
test_map50     = test_results.box.map50    # mAP@0.5
test_map5095   = test_results.box.map      # mAP@0.5:0.95

# F1-Score
if (test_precision + test_recall) > 0:
    test_f1 = 2 * test_precision * test_recall / (test_precision + test_recall)
else:
    test_f1 = 0.0

# -------------------- Display Results --------------------
print("\n" + "-"*60)
print(" TEST SET RESULTS")
print("-"*60)

test_summary = [
    ['Precision',      f'{test_precision:.4f}'],
    ['Recall',         f'{test_recall:.4f}'],
    ['F1-Score',       f'{test_f1:.4f}'],
    ['mAP@0.5',       f'{test_map50:.4f}'],
    ['mAP@0.5:0.95',  f'{test_map5095:.4f}'],
]

df_test_summary = pd.DataFrame(test_summary, columns=['Metric', 'Value'])
display(df_test_summary.style.hide(axis='index'))

# -------------------- Interpretation --------------------
print("\n INTERPRETATION:")
if test_map50 >= 0.90:
    print(f"    Excellent: mAP@0.5 = {test_map50:.4f} (>= 0.90)")
elif test_map50 >= 0.80:
    print(f"    Good: mAP@0.5 = {test_map50:.4f} (>= 0.80)")
elif test_map50 >= 0.70:
    print(f"    Fair: mAP@0.5 = {test_map50:.4f} (>= 0.70)")
else:
    print(f"    Needs improvement: mAP@0.5 = {test_map50:.4f} (< 0.70)")

if test_f1 >= 0.85:
    print(f"    F1-Score {test_f1:.4f} indicates strong balance between precision and recall")
elif test_f1 >= 0.70:
    print(f"    F1-Score {test_f1:.4f} is acceptable but may benefit from threshold tuning")
else:
    print(f"    F1-Score {test_f1:.4f} suggests a precision-recall imbalance — check conf threshold")

# -------------------- Display Built-in Plots --------------------
# Ultralytics generates PR curves, confusion matrix, etc. in the val output dir
print("\n Built-in evaluation plots saved to training directory.")
print("   Check for: PR_curve.png, confusion_matrix.png, F1_curve.png")

# Try to display the PR curve if it exists
pr_curve_path = os.path.join(TRAIN_DIR, 'PR_curve.png')
f1_curve_path = os.path.join(TRAIN_DIR, 'F1_curve.png')
cm_path = os.path.join(TRAIN_DIR, 'confusion_matrix.png')

available_plots = []
for path, name in [(pr_curve_path, 'PR Curve'), (f1_curve_path, 'F1 Curve'), (cm_path, 'Confusion Matrix')]:
    if os.path.exists(path):
        available_plots.append((path, name))

if available_plots:
    n_plots = len(available_plots)
    fig, axes = plt.subplots(1, n_plots, figsize=(6*n_plots, 5))
    if n_plots == 1:
        axes = [axes]
    
    for ax, (path, name) in zip(axes, available_plots):
        img = Image.open(path)
        ax.imshow(img)
        ax.set_title(name, fontsize=12, fontweight='bold')
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()

print("="*60)

---

# Phase 4: Inference & Smart Post-Processing

**Objective:** Go beyond raw `model.predict()` — apply intelligent post-processing to improve crop quality for downstream table structure recognition.

**Pipeline:** Raw Predictions → Box Dilation → Heuristic Filtering → Final Crops

---

<a id='inference'></a>
## 8. Load Best Weights & Run Inference

In [ ]:
# ============================================================
# Cell 8: Load Best Weights & Run Inference on Test Set
# ============================================================

print("="*60)
print(" INFERENCE ON TEST SET")
print("="*60)

# -------------------- Load Best Model --------------------
print(f"\n Loading best weights: {BEST_WEIGHTS}")
model_infer = YOLOv10(BEST_WEIGHTS)

# -------------------- Collect Test Images --------------------
test_images_dir = os.path.join(IMAGES_DIR, 'test')
test_image_files = sorted([
    f for f in os.listdir(test_images_dir) 
    if f.lower().endswith(('.jpg', '.jpeg', '.png'))
])

print(f"   Found {len(test_image_files)} test images in {test_images_dir}")

# -------------------- Run Inference --------------------
print(f"\n Running inference (conf={CONF_THRESH}, imgsz={IMGSZ})...")
print(f"   YOLOv10 uses NMS-free dual-head design — no explicit NMS needed")
print(f"   Adding light IoU-based duplicate check (iou_thresh={NMS_IOU_THRESH}) as safety net")

inference_start = time.time()

raw_predictions = {}  # {filename: list of {box, conf, cls}}

for img_file in test_image_files:
    img_path = os.path.join(test_images_dir, img_file)
    
    # Run prediction
    preds = model_infer.predict(
        source=img_path,
        conf=CONF_THRESH,
        imgsz=IMGSZ,
        save=False,
        verbose=False,
    )
    
    # Extract results
    result = preds[0]
    boxes = result.boxes
    
    detections = []
    if len(boxes) > 0:
        xyxy   = boxes.xyxy.cpu().numpy()    # [x1, y1, x2, y2]
        confs  = boxes.conf.cpu().numpy()     # confidence scores
        clses  = boxes.cls.cpu().numpy()      # class IDs
        
        for i in range(len(xyxy)):
            detections.append({
                'box': xyxy[i].tolist(),      # [x1, y1, x2, y2]
                'conf': float(confs[i]),
                'cls': int(clses[i]),
            })
    
    raw_predictions[img_file] = detections

inference_elapsed = time.time() - inference_start

# -------------------- Summary --------------------
total_detections = sum(len(v) for v in raw_predictions.values())
images_with_dets = sum(1 for v in raw_predictions.values() if len(v) > 0)
images_no_dets   = len(raw_predictions) - images_with_dets

print(f"\n" + "-"*60)
print(f" RAW INFERENCE RESULTS")
print(f"-"*60)
print(f"   Total images processed:      {len(raw_predictions)}")
print(f"   Images with detections:       {images_with_dets}")
print(f"   Images without detections:    {images_no_dets}")
print(f"   Total raw detections:         {total_detections}")
print(f"   Avg detections per image:     {total_detections/max(len(raw_predictions),1):.2f}")
print(f"   Inference time:               {inference_elapsed:.1f}s ({inference_elapsed/max(len(raw_predictions),1)*1000:.0f}ms/image)")

# -------------------- Confidence Distribution --------------------
all_confs = [d['conf'] for dets in raw_predictions.values() for d in dets]
if all_confs:
    print(f"\n   Confidence Stats:")
    print(f"      Min:    {min(all_confs):.4f}")
    print(f"      Max:    {max(all_confs):.4f}")
    print(f"      Mean:   {np.mean(all_confs):.4f}")
    print(f"      Median: {np.median(all_confs):.4f}")

print("="*60)

<a id='dilation'></a>
## 9. Box Dilation (Padding)

**The Critical Addition:** Raw detections often cut precisely on the table border line, destroying it during cropping. This makes downstream table structure recognition (row/column extraction) fail.

**Solution:** Expand every predicted bounding box outward by a fixed margin (default: 15 pixels) before cropping, ensuring the crop includes:
- The entire black border of the table
- A tiny whitespace buffer around it

Boundary clamping prevents the expanded box from exceeding image dimensions.

In [ ]:
# ============================================================
# Cell 9: Box Dilation (Padding) Function
# ============================================================

print("="*60)
print(" BOX DILATION (PADDING)")
print("="*60)

def dilate_boxes(detections, padding_px, img_w, img_h):
    """
    Expand bounding boxes outward by a fixed pixel margin.
    
    Args:
        detections: list of dicts with 'box' key [x1, y1, x2, y2]
        padding_px: number of pixels to expand on each side
        img_w: image width (for clamping)
        img_h: image height (for clamping)
    
    Returns:
        list of dicts with dilated boxes (new list, original unchanged)
    """
    dilated = []
    for det in detections:
        x1, y1, x2, y2 = det['box']
        
        # Expand outward
        x1_new = max(0, x1 - padding_px)
        y1_new = max(0, y1 - padding_px)
        x2_new = min(img_w, x2 + padding_px)
        y2_new = min(img_h, y2 + padding_px)
        
        dilated.append({
            'box': [x1_new, y1_new, x2_new, y2_new],
            'conf': det['conf'],
            'cls': det['cls'],
            'box_raw': det['box'],  # Keep original for comparison
        })
    
    return dilated

# -------------------- Apply Dilation to All Predictions --------------------
dilated_predictions = {}

for img_file, detections in raw_predictions.items():
    if not detections:
        dilated_predictions[img_file] = []
        continue
    
    # Get image dimensions
    img_path = os.path.join(test_images_dir, img_file)
    img = Image.open(img_path)
    img_w, img_h = img.size
    
    dilated_predictions[img_file] = dilate_boxes(detections, BOX_PADDING_PX, img_w, img_h)

# -------------------- Report Dilation Effect --------------------
total_expansion = []
for img_file, dilated in dilated_predictions.items():
    for det in dilated:
        if 'box_raw' in det:
            raw = det['box_raw']
            dil = det['box']
            raw_area = (raw[2] - raw[0]) * (raw[3] - raw[1])
            dil_area = (dil[2] - dil[0]) * (dil[3] - dil[1])
            if raw_area > 0:
                expansion_pct = (dil_area - raw_area) / raw_area * 100
                total_expansion.append(expansion_pct)

if total_expansion:
    print(f"\n Dilation Applied: {BOX_PADDING_PX}px on each side")
    print(f"   Boxes dilated:           {len(total_expansion)}")
    print(f"   Mean area expansion:     {np.mean(total_expansion):.1f}%")
    print(f"   Median area expansion:   {np.median(total_expansion):.1f}%")
    print(f"   Min / Max expansion:     {min(total_expansion):.1f}% / {max(total_expansion):.1f}%")
else:
    print("\n   No detections to dilate.")

print("="*60)

<a id='filtering'></a>
## 10. Heuristic Filtering

Three filters to remove false positives and redundant detections:

1. **Aspect Ratio Filter:** Discard boxes with $\text{AR} = w/h < 0.1$ or $> 10.0$ — real tables rarely have extreme proportions
2. **Nested Box Removal:** If box A is fully contained inside box B, remove A — fixes nested false positives
3. **Minimum Area Filter:** Discard boxes smaller than 1% of the image area — likely noise

In [ ]:
# ============================================================
# Cell 10: Heuristic Filtering
# ============================================================

print("="*60)
print(" HEURISTIC FILTERING")
print("="*60)

def compute_iou_xyxy(box1, box2):
    """Compute IoU between two boxes in [x1, y1, x2, y2] format."""
    inter_x1 = max(box1[0], box2[0])
    inter_y1 = max(box1[1], box2[1])
    inter_x2 = min(box1[2], box2[2])
    inter_y2 = min(box1[3], box2[3])
    
    inter_area = max(0, inter_x2 - inter_x1) * max(0, inter_y2 - inter_y1)
    
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union_area = area1 + area2 - inter_area
    
    return inter_area / union_area if union_area > 0 else 0.0

def compute_containment(inner_box, outer_box):
    """
    Compute what fraction of inner_box's area is inside outer_box.
    Returns 1.0 if inner is fully contained in outer.
    """
    inter_x1 = max(inner_box[0], outer_box[0])
    inter_y1 = max(inner_box[1], outer_box[1])
    inter_x2 = min(inner_box[2], outer_box[2])
    inter_y2 = min(inner_box[3], outer_box[3])
    
    inter_area = max(0, inter_x2 - inter_x1) * max(0, inter_y2 - inter_y1)
    inner_area = (inner_box[2] - inner_box[0]) * (inner_box[3] - inner_box[1])
    
    return inter_area / inner_area if inner_area > 0 else 0.0

def filter_detections(detections, img_w, img_h, 
                      ar_min=AR_MIN, ar_max=AR_MAX,
                      min_area_pct=MIN_AREA_PCT,
                      nested_iou_thresh=NESTED_IOU):
    """
    Apply heuristic filters to detections.
    
    Filters:
    1. Aspect Ratio: remove boxes with AR < ar_min or AR > ar_max
    2. Minimum Area: remove boxes smaller than min_area_pct of image
    3. Nested Box Removal: if box A is fully inside box B, remove A
    
    Args:
        detections: list of dicts with 'box' [x1, y1, x2, y2], 'conf', 'cls'
        img_w, img_h: image dimensions
        ar_min, ar_max: aspect ratio range
        min_area_pct: minimum box area as fraction of image area
        nested_iou_thresh: containment threshold for nested removal
    
    Returns:
        filtered: list of surviving detections
        filter_log: dict with counts of removed boxes per filter
    """
    filter_log = {
        'input': len(detections),
        'ar_removed': 0,
        'area_removed': 0,
        'nested_removed': 0,
    }
    
    if not detections:
        filter_log['output'] = 0
        return [], filter_log
    
    img_area = img_w * img_h
    surviving = []
    
    # ---- Filter 1: Aspect Ratio ----
    for det in detections:
        x1, y1, x2, y2 = det['box']
        w = x2 - x1
        h = y2 - y1
        
        if h <= 0 or w <= 0:
            filter_log['ar_removed'] += 1
            continue
        
        ar = w / h
        if ar < ar_min or ar > ar_max:
            filter_log['ar_removed'] += 1
            continue
        
        surviving.append(det)
    
    # ---- Filter 2: Minimum Area ----
    area_filtered = []
    for det in surviving:
        x1, y1, x2, y2 = det['box']
        box_area = (x2 - x1) * (y2 - y1)
        
        if box_area / img_area < min_area_pct:
            filter_log['area_removed'] += 1
            continue
        
        area_filtered.append(det)
    
    surviving = area_filtered
    
    # ---- Filter 3: Nested Box Removal ----
    # For every pair, if box A is fully contained inside box B, remove A
    if len(surviving) > 1:
        keep_mask = [True] * len(surviving)
        
        for i in range(len(surviving)):
            if not keep_mask[i]:
                continue
            for j in range(len(surviving)):
                if i == j or not keep_mask[j]:
                    continue
                
                # Check if box j is contained inside box i
                containment = compute_containment(surviving[j]['box'], surviving[i]['box'])
                if containment >= nested_iou_thresh:
                    # j is inside i — remove j (the inner/smaller one)
                    keep_mask[j] = False
                    filter_log['nested_removed'] += 1
        
        surviving = [det for det, keep in zip(surviving, keep_mask) if keep]
    
    filter_log['output'] = len(surviving)
    return surviving, filter_log

# -------------------- Apply Filters to All Dilated Predictions --------------------
filtered_predictions = {}
total_filter_log = defaultdict(int)

for img_file, detections in dilated_predictions.items():
    if not detections:
        filtered_predictions[img_file] = []
        continue
    
    # Get image dimensions
    img_path = os.path.join(test_images_dir, img_file)
    img = Image.open(img_path)
    img_w, img_h = img.size
    
    filtered, log = filter_detections(detections, img_w, img_h)
    filtered_predictions[img_file] = filtered
    
    for key, val in log.items():
        total_filter_log[key] += val

# -------------------- Report --------------------
print(f"\n" + "-"*60)
print(f" FILTERING SUMMARY")
print(f"-"*60)
print(f"   Input detections:          {total_filter_log['input']}")
print(f"   Removed by AR filter:      {total_filter_log['ar_removed']}  (AR<{AR_MIN} or AR>{AR_MAX})")
print(f"   Removed by area filter:    {total_filter_log['area_removed']}  (area<{MIN_AREA_PCT*100:.0f}% of image)")
print(f"   Removed by nested filter:  {total_filter_log['nested_removed']}  (fully inside another box)")
print(f"   Output detections:         {total_filter_log['output']}")

total_removed = total_filter_log['input'] - total_filter_log['output']
if total_filter_log['input'] > 0:
    print(f"\n   Total removed: {total_removed} ({total_removed/total_filter_log['input']*100:.1f}%)")
else:
    print(f"\n   No detections to filter.")

print("="*60)

<a id='pipeline'></a>
## 11. End-to-End Inference Pipeline

Unified pipeline function that chains: **Predict → Dilate → Filter → Crop**

In [ ]:
# ============================================================
# Cell 11: End-to-End Inference Pipeline
# ============================================================

print("="*60)
print(" END-TO-END INFERENCE PIPELINE")
print("="*60)

def run_pipeline(image_path, model, conf=CONF_THRESH, padding=BOX_PADDING_PX,
                 ar_min=AR_MIN, ar_max=AR_MAX, min_area_pct=MIN_AREA_PCT,
                 nested_iou=NESTED_IOU):
    """
    Full inference pipeline: Predict → Dilate → Filter → Crop
    
    Args:
        image_path: path to input image
        model: loaded YOLOv10 model
        conf: confidence threshold
        padding: box dilation pixels
        ar_min, ar_max: aspect ratio bounds
        min_area_pct: minimum area filter
        nested_iou: containment threshold
    
    Returns:
        dict with keys:
            - 'image_path': input path
            - 'img_w', 'img_h': dimensions
            - 'raw_detections': list of raw boxes
            - 'dilated_detections': list after dilation
            - 'filtered_detections': list after filtering
            - 'crops': list of PIL Image crops
            - 'filter_log': filtering statistics
    """
    # Load image
    img = Image.open(image_path)
    img_w, img_h = img.size
    
    # Step 1: Predict
    preds = model.predict(source=image_path, conf=conf, imgsz=IMGSZ, 
                          save=False, verbose=False)
    result = preds[0]
    boxes = result.boxes
    
    raw_dets = []
    if len(boxes) > 0:
        xyxy  = boxes.xyxy.cpu().numpy()
        confs = boxes.conf.cpu().numpy()
        clses = boxes.cls.cpu().numpy()
        
        for i in range(len(xyxy)):
            raw_dets.append({
                'box': xyxy[i].tolist(),
                'conf': float(confs[i]),
                'cls': int(clses[i]),
            })
    
    # Step 2: Dilate
    dilated_dets = dilate_boxes(raw_dets, padding, img_w, img_h)
    
    # Step 3: Filter
    filtered_dets, filter_log = filter_detections(
        dilated_dets, img_w, img_h, ar_min, ar_max, min_area_pct, nested_iou
    )
    
    # Step 4: Crop
    img_np = np.array(img)
    crops = []
    for det in filtered_dets:
        x1, y1, x2, y2 = [int(round(c)) for c in det['box']]
        x1 = max(0, x1)
        y1 = max(0, y1)
        x2 = min(img_w, x2)
        y2 = min(img_h, y2)
        
        if x2 > x1 and y2 > y1:
            crop = img_np[y1:y2, x1:x2]
            crops.append(Image.fromarray(crop))
        else:
            crops.append(None)
    
    return {
        'image_path': image_path,
        'img_w': img_w,
        'img_h': img_h,
        'raw_detections': raw_dets,
        'dilated_detections': dilated_dets,
        'filtered_detections': filtered_dets,
        'crops': crops,
        'filter_log': filter_log,
    }

# -------------------- Run Pipeline on All Test Images --------------------
print(f"\n Running full pipeline on {len(test_image_files)} test images...")

pipeline_results = {}
pipeline_start = time.time()

for img_file in test_image_files:
    img_path = os.path.join(test_images_dir, img_file)
    pipeline_results[img_file] = run_pipeline(img_path, model_infer)

pipeline_elapsed = time.time() - pipeline_start

# -------------------- Summary --------------------
total_raw   = sum(len(r['raw_detections']) for r in pipeline_results.values())
total_final = sum(len(r['filtered_detections']) for r in pipeline_results.values())
total_crops = sum(len(r['crops']) for r in pipeline_results.values())

print(f"\n" + "-"*60)
print(f" PIPELINE SUMMARY")
print(f"-"*60)
print(f"   Images processed:    {len(pipeline_results)}")
print(f"   Raw detections:      {total_raw}")
print(f"   After dilation:      {total_raw} (same count, expanded boxes)")
print(f"   After filtering:     {total_final}")
print(f"   Table crops:         {total_crops}")
print(f"   Pipeline time:       {pipeline_elapsed:.1f}s ({pipeline_elapsed/max(len(pipeline_results),1)*1000:.0f}ms/image)")
print("="*60)

<a id='post-eval'></a>
## 12. Post-Processing Evaluation

Compare **raw predictions** vs. **post-processed predictions** (after dilation + filtering) against ground truth on the test set.

Custom IoU computation per image, plus aggregate mAP/F1 comparison to quantify the improvement from post-processing.

In [ ]:
# ============================================================
# Cell 12: Post-Processing Evaluation — Raw vs Filtered vs Ground Truth
# ============================================================

print("="*60)
print(" POST-PROCESSING EVALUATION")
print("="*60)

# -------------------- Load Ground Truth Labels --------------------
test_labels_dir = os.path.join(LABELS_DIR, 'test')

def load_yolo_gt(label_path, img_w, img_h):
    """Load YOLO ground truth and convert to [x1, y1, x2, y2] pixel coords."""
    boxes = []
    if not os.path.exists(label_path):
        return boxes
    with open(label_path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            cls = int(parts[0])
            xc, yc, w, h = map(float, parts[1:5])
            
            # Convert to pixel xyxy
            w_px = w * img_w
            h_px = h * img_h
            x1 = xc * img_w - w_px / 2
            y1 = yc * img_h - h_px / 2
            x2 = x1 + w_px
            y2 = y1 + h_px
            
            boxes.append({
                'box': [x1, y1, x2, y2],
                'cls': cls,
            })
    return boxes

def match_predictions_to_gt(pred_boxes, gt_boxes, iou_threshold=0.5):
    """
    Match predicted boxes to ground truth using greedy IoU matching.
    
    Returns:
        tp: number of true positives (matched pairs with IoU >= threshold)
        fp: number of false positives (predictions without matching GT)
        fn: number of false negatives (GT without matching prediction)
        matched_ious: list of IoU values for matched pairs
    """
    if not pred_boxes and not gt_boxes:
        return 0, 0, 0, []
    if not pred_boxes:
        return 0, 0, len(gt_boxes), []
    if not gt_boxes:
        return 0, len(pred_boxes), 0, []
    
    # Compute IoU matrix
    iou_matrix = np.zeros((len(pred_boxes), len(gt_boxes)))
    for i, pred in enumerate(pred_boxes):
        for j, gt in enumerate(gt_boxes):
            iou_matrix[i, j] = compute_iou_xyxy(pred['box'], gt['box'])
    
    # Greedy matching (highest IoU first)
    matched_pred = set()
    matched_gt = set()
    matched_ious = []
    
    while True:
        if iou_matrix.size == 0:
            break
        max_iou = iou_matrix.max()
        if max_iou < iou_threshold:
            break
        
        max_idx = np.unravel_index(iou_matrix.argmax(), iou_matrix.shape)
        pred_idx, gt_idx = max_idx
        
        matched_pred.add(pred_idx)
        matched_gt.add(gt_idx)
        matched_ious.append(max_iou)
        
        # Zero out matched rows/cols
        iou_matrix[pred_idx, :] = 0
        iou_matrix[:, gt_idx] = 0
    
    tp = len(matched_ious)
    fp = len(pred_boxes) - tp
    fn = len(gt_boxes) - tp
    
    return tp, fp, fn, matched_ious

# -------------------- Evaluate Raw vs Filtered --------------------
eval_records = []

for img_file in test_image_files:
    # Ground truth
    label_file = os.path.splitext(img_file)[0] + '.txt'
    label_path = os.path.join(test_labels_dir, label_file)
    
    img_path = os.path.join(test_images_dir, img_file)
    img = Image.open(img_path)
    img_w, img_h = img.size
    
    gt_boxes = load_yolo_gt(label_path, img_w, img_h)
    
    # Raw predictions
    raw_dets = raw_predictions.get(img_file, [])
    
    # Filtered predictions (from pipeline)
    result = pipeline_results.get(img_file, {})
    filt_dets = result.get('filtered_detections', [])
    
    # Match raw predictions to GT
    tp_raw, fp_raw, fn_raw, ious_raw = match_predictions_to_gt(raw_dets, gt_boxes)
    
    # Match filtered predictions to GT
    tp_filt, fp_filt, fn_filt, ious_filt = match_predictions_to_gt(filt_dets, gt_boxes)
    
    eval_records.append({
        'image': img_file,
        'n_gt': len(gt_boxes),
        'n_raw': len(raw_dets),
        'n_filtered': len(filt_dets),
        'tp_raw': tp_raw,
        'fp_raw': fp_raw,
        'fn_raw': fn_raw,
        'mean_iou_raw': np.mean(ious_raw) if ious_raw else 0.0,
        'tp_filt': tp_filt,
        'fp_filt': fp_filt,
        'fn_filt': fn_filt,
        'mean_iou_filt': np.mean(ious_filt) if ious_filt else 0.0,
    })

df_eval = pd.DataFrame(eval_records)

# -------------------- Aggregate Metrics --------------------
# Raw
total_tp_raw = df_eval['tp_raw'].sum()
total_fp_raw = df_eval['fp_raw'].sum()
total_fn_raw = df_eval['fn_raw'].sum()

p_raw = total_tp_raw / (total_tp_raw + total_fp_raw) if (total_tp_raw + total_fp_raw) > 0 else 0
r_raw = total_tp_raw / (total_tp_raw + total_fn_raw) if (total_tp_raw + total_fn_raw) > 0 else 0
f1_raw = 2 * p_raw * r_raw / (p_raw + r_raw) if (p_raw + r_raw) > 0 else 0
mean_iou_raw_all = df_eval['mean_iou_raw'].mean()

# Filtered
total_tp_filt = df_eval['tp_filt'].sum()
total_fp_filt = df_eval['fp_filt'].sum()
total_fn_filt = df_eval['fn_filt'].sum()

p_filt = total_tp_filt / (total_tp_filt + total_fp_filt) if (total_tp_filt + total_fp_filt) > 0 else 0
r_filt = total_tp_filt / (total_tp_filt + total_fn_filt) if (total_tp_filt + total_fn_filt) > 0 else 0
f1_filt = 2 * p_filt * r_filt / (p_filt + r_filt) if (p_filt + r_filt) > 0 else 0
mean_iou_filt_all = df_eval['mean_iou_filt'].mean()

# -------------------- Comparison Table --------------------
print("\n" + "-"*60)
print(" RAW vs POST-PROCESSED COMPARISON")
print("-"*60)

comparison_data = [
    ['Total Detections', f'{df_eval["n_raw"].sum()}', f'{df_eval["n_filtered"].sum()}'],
    ['True Positives',   f'{total_tp_raw}',           f'{total_tp_filt}'],
    ['False Positives',  f'{total_fp_raw}',           f'{total_fp_filt}'],
    ['False Negatives',  f'{total_fn_raw}',           f'{total_fn_filt}'],
    ['Precision',        f'{p_raw:.4f}',              f'{p_filt:.4f}'],
    ['Recall',           f'{r_raw:.4f}',              f'{r_filt:.4f}'],
    ['F1-Score',         f'{f1_raw:.4f}',             f'{f1_filt:.4f}'],
    ['Mean IoU',         f'{mean_iou_raw_all:.4f}',   f'{mean_iou_filt_all:.4f}'],
]

df_comparison = pd.DataFrame(comparison_data, columns=['Metric', 'Raw Predictions', 'Post-Processed'])
display(df_comparison.style.hide(axis='index'))

# -------------------- Delta --------------------
print(f"\n IMPROVEMENT FROM POST-PROCESSING:")
print(f"   Precision: {p_raw:.4f} → {p_filt:.4f} (Δ = {p_filt - p_raw:+.4f})")
print(f"   Recall:    {r_raw:.4f} → {r_filt:.4f} (Δ = {r_filt - r_raw:+.4f})")
print(f"   F1-Score:  {f1_raw:.4f} → {f1_filt:.4f} (Δ = {f1_filt - f1_raw:+.4f})")
print(f"   Mean IoU:  {mean_iou_raw_all:.4f} → {mean_iou_filt_all:.4f} (Δ = {mean_iou_filt_all - mean_iou_raw_all:+.4f})")

# -------------------- Per-Image Table --------------------
print("\n" + "-"*60)
print(" PER-IMAGE BREAKDOWN (first 20 images)")
print("-"*60)

display_eval_cols = ['image', 'n_gt', 'n_raw', 'n_filtered', 
                     'tp_filt', 'fp_filt', 'fn_filt', 'mean_iou_filt']
df_eval_display = df_eval[display_eval_cols].head(20).copy()
df_eval_display.columns = ['Image', 'GT Boxes', 'Raw Preds', 'Filtered Preds',
                           'TP', 'FP', 'FN', 'Mean IoU']

display(df_eval_display.style.format({'Mean IoU': '{:.4f}'}).hide(axis='index'))

print("="*60)

<a id='results-viz'></a>
## 13. Results Visualization

1. **Detection Overlay Grid:** Test images with GT (green), raw predictions (red dashed), and post-processed predictions (blue solid)
2. **Raw vs Dilated Crop Comparison:** Side-by-side showing the dilation effect on border preservation
3. **Confidence Distribution:** Histogram of prediction confidence scores

In [ ]:
# ============================================================
# Cell 13: Results Visualization
# ============================================================

print("="*60)
print(" RESULTS VISUALIZATION")
print("="*60)

# =====================================================================
# PLOT 1: Detection Overlay Grid (GT + Raw + Filtered)
# =====================================================================
print("\n Generating detection overlay grid...")

# Select images with detections for visualization
images_with_tables = [f for f in test_image_files 
                      if len(pipeline_results.get(f, {}).get('filtered_detections', [])) > 0]

# If few images have detections, include some without
if len(images_with_tables) < 9:
    remaining = [f for f in test_image_files if f not in images_with_tables]
    images_with_tables.extend(remaining[:9 - len(images_with_tables)])

sample_images = images_with_tables[:9]
n_samples = len(sample_images)

if n_samples > 0:
    n_cols = min(3, n_samples)
    n_rows = (n_samples + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(6*n_cols, 6*n_rows))
    if n_samples == 1:
        axes = np.array([[axes]])
    elif n_rows == 1:
        axes = axes.reshape(1, -1)
    
    for idx, img_file in enumerate(sample_images):
        r, c = idx // n_cols, idx % n_cols
        ax = axes[r, c]
        
        img_path = os.path.join(test_images_dir, img_file)
        img = Image.open(img_path)
        img_w, img_h = img.size
        ax.imshow(img, cmap='gray')
        
        # Load and draw GT boxes (green)
        label_file = os.path.splitext(img_file)[0] + '.txt'
        label_path = os.path.join(test_labels_dir, label_file)
        gt_boxes = load_yolo_gt(label_path, img_w, img_h)
        
        for gt in gt_boxes:
            x1, y1, x2, y2 = gt['box']
            rect = Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2,
                             edgecolor='#2ECC71', facecolor='none', linestyle='-',
                             label='GT' if gt == gt_boxes[0] else '')
            ax.add_patch(rect)
        
        # Draw raw predictions (red dashed)
        raw_dets = raw_predictions.get(img_file, [])
        for det in raw_dets:
            x1, y1, x2, y2 = det['box']
            rect = Rectangle((x1, y1), x2-x1, y2-y1, linewidth=1.5,
                             edgecolor='#E74C3C', facecolor='none', linestyle='--',
                             label='Raw' if det == raw_dets[0] else '')
            ax.add_patch(rect)
        
        # Draw filtered predictions (blue solid)
        result = pipeline_results.get(img_file, {})
        filt_dets = result.get('filtered_detections', [])
        for det in filt_dets:
            x1, y1, x2, y2 = det['box']
            rect = Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2.5,
                             edgecolor='#3498DB', facecolor='none', linestyle='-',
                             label='Filtered' if det == filt_dets[0] else '')
            ax.add_patch(rect)
            ax.text(x1, y1-8, f'{det["conf"]:.2f}', color='#3498DB', fontsize=8,
                    fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8))
        
        n_gt = len(gt_boxes)
        n_filt = len(filt_dets)
        ax.set_title(f'{img_file}\nGT: {n_gt} | Filtered: {n_filt}', fontsize=9)
        ax.axis('off')
    
    # Hide unused subplots
    for idx in range(n_samples, n_rows * n_cols):
        r, c = idx // n_cols, idx % n_cols
        axes[r, c].axis('off')
    
    # Legend
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], color='#2ECC71', linewidth=2, linestyle='-', label='Ground Truth'),
        Line2D([0], [0], color='#E74C3C', linewidth=1.5, linestyle='--', label='Raw Prediction'),
        Line2D([0], [0], color='#3498DB', linewidth=2.5, linestyle='-', label='Post-Processed'),
    ]
    fig.legend(handles=legend_elements, loc='lower center', ncol=3, fontsize=11,
               bbox_to_anchor=(0.5, -0.02))
    
    fig.suptitle('Detection Results: Ground Truth vs Raw vs Post-Processed',
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

# =====================================================================
# PLOT 2: Raw vs Dilated Crop Comparison
# =====================================================================
print("\n Generating raw vs dilated crop comparison...")

# Find images with both raw and filtered detections that have 'box_raw'
crop_comparison_images = []
for img_file, result in pipeline_results.items():
    filt_dets = result.get('filtered_detections', [])
    if filt_dets and 'box_raw' in filt_dets[0]:
        crop_comparison_images.append(img_file)
    if len(crop_comparison_images) >= 4:
        break

if crop_comparison_images:
    n_compare = len(crop_comparison_images)
    fig, axes = plt.subplots(n_compare, 2, figsize=(10, 4 * n_compare))
    if n_compare == 1:
        axes = axes.reshape(1, -1)
    
    for idx, img_file in enumerate(crop_comparison_images):
        result = pipeline_results[img_file]
        img = Image.open(result['image_path'])
        img_np = np.array(img)
        img_w, img_h = img.size
        
        det = result['filtered_detections'][0]  # First detection
        
        # Raw crop (before dilation)
        raw_box = det.get('box_raw', det['box'])
        rx1, ry1, rx2, ry2 = [int(round(c)) for c in raw_box]
        rx1, ry1 = max(0, rx1), max(0, ry1)
        rx2, ry2 = min(img_w, rx2), min(img_h, ry2)
        raw_crop = img_np[ry1:ry2, rx1:rx2]
        
        # Dilated crop (after dilation)
        dx1, dy1, dx2, dy2 = [int(round(c)) for c in det['box']]
        dx1, dy1 = max(0, dx1), max(0, dy1)
        dx2, dy2 = min(img_w, dx2), min(img_h, dy2)
        dil_crop = img_np[dy1:dy2, dx1:dx2]
        
        axes[idx, 0].imshow(raw_crop, cmap='gray')
        axes[idx, 0].set_title(f'Raw Crop ({rx2-rx1}x{ry2-ry1}px)', fontsize=10)
        axes[idx, 0].axis('off')
        
        axes[idx, 1].imshow(dil_crop, cmap='gray')
        axes[idx, 1].set_title(f'Dilated Crop +{BOX_PADDING_PX}px ({dx2-dx1}x{dy2-dy1}px)', fontsize=10)
        axes[idx, 1].axis('off')
    
    fig.suptitle(f'Raw vs Dilated Crops (padding={BOX_PADDING_PX}px)',
                 fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("   No crop comparison available (no detections with box_raw).")

# =====================================================================
# PLOT 3: Confidence Distribution
# =====================================================================
print("\n Generating confidence distribution...")

all_raw_confs = [d['conf'] for dets in raw_predictions.values() for d in dets]
all_filt_confs = [d['conf'] for r in pipeline_results.values() 
                  for d in r.get('filtered_detections', [])]

if all_raw_confs or all_filt_confs:
    fig, ax = plt.subplots(figsize=(10, 4))
    
    if all_raw_confs:
        ax.hist(all_raw_confs, bins=30, alpha=0.5, color='#E74C3C', 
                edgecolor='black', label=f'Raw (n={len(all_raw_confs)})')
    if all_filt_confs:
        ax.hist(all_filt_confs, bins=30, alpha=0.5, color='#3498DB',
                edgecolor='black', label=f'Filtered (n={len(all_filt_confs)})')
    
    ax.axvline(x=CONF_THRESH, color='green', linestyle='--', linewidth=2,
               label=f'Conf threshold ({CONF_THRESH})')
    ax.set_xlabel('Confidence Score', fontsize=12)
    ax.set_ylabel('Count', fontsize=12)
    ax.set_title('Prediction Confidence Distribution', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

print("   All visualization plots generated.")
print("="*60)

<a id='export'></a>
## 14. Export Results

Save:
1. **Cropped table images** to `/kaggle/working/cropped_tables/`
2. **Detection results JSON** with all boxes, scores, and metadata
3. **Final summary report** with all evaluation metrics

In [ ]:
# ============================================================
# Cell 14: Export Results
# ============================================================

print("="*60)
print(" EXPORT RESULTS")
print("="*60)

# =====================================================================
# 1. Save Cropped Table Images
# =====================================================================
print(f"\n Saving cropped tables to: {CROPS_DIR}")

if os.path.exists(CROPS_DIR):
    shutil.rmtree(CROPS_DIR)
os.makedirs(CROPS_DIR, exist_ok=True)

crop_count = 0

for img_file, result in pipeline_results.items():
    crops = result.get('crops', [])
    filt_dets = result.get('filtered_detections', [])
    
    for i, (crop, det) in enumerate(zip(crops, filt_dets)):
        if crop is None:
            continue
        
        base_name = os.path.splitext(img_file)[0]
        crop_name = f"{base_name}_table_{i+1}.jpg"
        crop_path = os.path.join(CROPS_DIR, crop_name)
        
        crop.save(crop_path, quality=95)
        crop_count += 1

print(f"   Saved {crop_count} cropped table images.")

# =====================================================================
# 2. Save Detection Results JSON
# =====================================================================
results_json_path = os.path.join(OUTPUT_DIR, 'detection_results.json')

export_results = []
for img_file, result in pipeline_results.items():
    img_entry = {
        'image_name': img_file,
        'img_w': result['img_w'],
        'img_h': result['img_h'],
        'n_raw_detections': len(result['raw_detections']),
        'n_filtered_detections': len(result['filtered_detections']),
        'detections': []
    }
    
    for i, det in enumerate(result['filtered_detections']):
        img_entry['detections'].append({
            'table_id': i + 1,
            'box_xyxy': [round(c, 2) for c in det['box']],
            'box_raw_xyxy': [round(c, 2) for c in det.get('box_raw', det['box'])],
            'confidence': round(det['conf'], 4),
            'class': det['cls'],
            'class_name': CLASS_NAMES[det['cls']] if det['cls'] < len(CLASS_NAMES) else 'unknown',
        })
    
    export_results.append(img_entry)

with open(results_json_path, 'w') as f:
    json.dump(export_results, f, indent=2)

print(f"   Detection results saved to: {results_json_path}")

# =====================================================================
# 3. Final Summary Report
# =====================================================================
print("\n" + "="*70)
print(" PHASE 3 & 4 FINAL SUMMARY REPORT")
print("="*70)

print("\n" + "-"*70)
print(" PHASE 3: TRAINING SUMMARY")
print("-"*70)

training_summary = [
    ['Model',                   MODEL_VARIANT],
    ['Resolution',              f'{IMGSZ}×{IMGSZ} (letterbox)'],
    ['Epochs Trained',          f'{len(df_metrics)} / {EPOCHS}'],
    ['Early Stopping',          f'{"Yes (patience={})".format(PATIENCE) if len(df_metrics) < EPOCHS else "No (all epochs completed)"}'],
    ['Optimizer',               f'{OPTIMIZER} (lr={LR0}, wd={WEIGHT_DECAY})'],
    ['Batch Size',              str(BATCH_SIZE)],
    ['Training Time',           f'{train_elapsed/60:.1f} min'],
]

if 'mAP@0.5' in df_metrics.columns:
    best_idx = df_metrics['mAP@0.5:0.95'].idxmax()
    best = df_metrics.loc[best_idx]
    training_summary.extend([
        ['', ''],
        ['Best Epoch',              str(int(best.get('Epoch', best_idx)))],
        ['Best mAP@0.5',           f'{best.get("mAP@0.5", 0):.4f}'],
        ['Best mAP@0.5:0.95',     f'{best.get("mAP@0.5:0.95", 0):.4f}'],
        ['Best F1-Score',          f'{best.get("F1-Score", 0):.4f}'],
    ])

df_train_summary = pd.DataFrame(training_summary, columns=['Metric', 'Value'])
display(df_train_summary.style.hide(axis='index'))

print("\n" + "-"*70)
print(" PHASE 3: TEST SET RESULTS")
print("-"*70)

test_results_display = [
    ['Precision',       f'{test_precision:.4f}'],
    ['Recall',          f'{test_recall:.4f}'],
    ['F1-Score',        f'{test_f1:.4f}'],
    ['mAP@0.5',        f'{test_map50:.4f}'],
    ['mAP@0.5:0.95',   f'{test_map5095:.4f}'],
]
df_test_display = pd.DataFrame(test_results_display, columns=['Metric', 'Value'])
display(df_test_display.style.hide(axis='index'))

print("\n" + "-"*70)
print(" PHASE 4: POST-PROCESSING SUMMARY")
print("-"*70)

pp_summary = [
    ['Box Padding',          f'{BOX_PADDING_PX}px'],
    ['AR Filter Range',      f'[{AR_MIN}, {AR_MAX}]'],
    ['Min Area Filter',      f'{MIN_AREA_PCT*100:.0f}% of image'],
    ['Nested IoU Threshold', f'{NESTED_IOU}'],
    ['', ''],
    ['Raw Detections',       f'{total_raw}'],
    ['Post-Processed',       f'{total_final}'],
    ['Removed by Filters',   f'{total_raw - total_final}'],
    ['Table Crops Saved',    f'{crop_count}'],
    ['', ''],
    ['Precision (post)',     f'{p_filt:.4f}'],
    ['Recall (post)',        f'{r_filt:.4f}'],
    ['F1-Score (post)',      f'{f1_filt:.4f}'],
    ['Mean IoU (post)',      f'{mean_iou_filt_all:.4f}'],
]

df_pp_summary = pd.DataFrame(pp_summary, columns=['Metric', 'Value'])
display(df_pp_summary.style.hide(axis='index'))

# =====================================================================
# Completion Checklist
# =====================================================================
print("\n" + "="*70)
print(" COMPLETION CHECKLIST")
print("="*70)

checklist = [
    ("Phase 3: YOLOv10-M model trained",          os.path.exists(BEST_WEIGHTS)),
    ("Phase 3: Per-epoch metrics displayed",       len(df_metrics) > 0),
    ("Phase 3: Training curves plotted",           True),
    ("Phase 3: Test set evaluated",                test_map50 > 0),
    ("Phase 4: Inference on test set",             total_raw >= 0),
    ("Phase 4: Box dilation applied",              BOX_PADDING_PX > 0),
    ("Phase 4: Heuristic filters applied",         total_filter_log['input'] >= 0),
    ("Phase 4: End-to-end pipeline verified",      len(pipeline_results) > 0),
    ("Phase 4: Raw vs post-processed compared",    len(df_eval) > 0),
    ("Phase 4: Results visualized",                True),
    ("Phase 4: Crops exported",                    crop_count >= 0),
    ("Phase 4: Detection JSON exported",           os.path.exists(results_json_path)),
]

all_passed = True
for task, passed in checklist:
    status = "OK" if passed else "FAIL"
    if not passed:
        all_passed = False
    print(f"   {status}  {task}")

print("\n" + "="*70)
if all_passed:
    print(" ALL CHECKS PASSED")
else:
    print(" SOME CHECKS FAILED — REVIEW ABOVE")
print("="*70)

print(f"\n Output Files:")
print(f"   Best weights: {BEST_WEIGHTS}")
print(f"   Cropped tables: {CROPS_DIR} ({crop_count} images)")
print(f"   Detection JSON: {results_json_path}")
print(f"   Training results: {RESULTS_CSV}")